In [2]:
import pandas as pd

s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")

for name, df in [("source1", s1), ("source2", s2), ("source3", s3), ("ground_truth", gt)]:
    print(name, df.shape)
    print(df.head(), "\n")
    

source1 (2206821, 4)
      entity_id            business_name  \
0  S1-925783039      Orelee's Barbershop   
1  S1-773889195              Prime Money   
2  S1-377745466            B+ Retail Inc   
3  S1-133037285            Christ Chapel   
4  S1-755362802  Prabhav Business Center   

                                    business_address country  
0             1795 Westchester Drive, High Point, NC      US  
1                    17560 Ellis Road, Tahlequah, OK      US  
2                1712 Montebello Avenue, Phoenix, AZ      US  
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US  
4  797, Lake Town Block A, Kolkata, Howrah, West ...   India   

source2 (5034616, 4)
      entity_id                    business_name  \
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1  S2-764573417     -- Holloway Peak Inc Seafood   
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी   
3  S2-163963287                       Summit Inc   
4   S2-49942811     Delta Tetlecommunication Inc   



In [8]:
def blocking_key(name, country):
    if pd.isna(name):
        return f"{country}_MISSING"
    clean = name.lower().strip()
    clean = ''.join(c for c in clean if c.isalnum())
    return f"{country}_{clean[:4]}"

In [9]:
print(s1['country'].value_counts())
print(s2['country'].value_counts())
print(s3['country'].value_counts())

country
US       1323633
India     883188
Name: count, dtype: int64
country
US       3016817
India    2017799
Name: count, dtype: int64
country
US       3170056
India    2115547
Name: count, dtype: int64


In [10]:
# apply blocking key to every row
s1['block_key'] = s1.apply(lambda r: blocking_key(r['business_name'], r['country']), axis=1)
s2['block_key'] = s2.apply(lambda r: blocking_key(r['business_name'], r['country']), axis=1)
s3['block_key'] = s3.apply(lambda r: blocking_key(r['business_name'], r['country']), axis=1)

# how many records land in each bucket, for s2 and s3?
print(s2['block_key'].value_counts().describe())
print(s3['block_key'].value_counts().describe())

count    226543.000000
mean         22.223666
std         273.103238
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max       39818.000000
Name: count, dtype: float64
count    229562.000000
mean         23.024730
std         293.836371
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max       46761.000000
Name: count, dtype: float64


In [11]:
print(s1['business_name'].isna().sum(), s1.shape[0])
print(s2['business_name'].isna().sum(), s2.shape[0])
print(s3['business_name'].isna().sum(), s3.shape[0])

0 2206821
2 5034616
13 5285603


In [12]:
print("SOURCE 2 - TOP 20 LARGEST BUCKETS")
print(
    s2["block_key"]
    .value_counts()
    .head(20)
)

print("\nSOURCE 3 - TOP 20 LARGEST BUCKETS")
print(
    s3["block_key"]
    .value_counts()
    .head(20)
)

SOURCE 2 - TOP 20 LARGEST BUCKETS
block_key
India_priv    39818
US_pedi       30732
US_inte       27623
India_shri    26765
US_prim       15516
US_metr       15271
India_limi    15079
India_indi    14030
US_fami       13736
US_card       12399
US_brig       12287
US_dent       11115
US_west       10930
India_tech    10884
US_phys       10824
US_stra       10666
US_blue       10581
US_corn       10578
US_derm       10505
US_foot       10408
Name: count, dtype: int64

SOURCE 3 - TOP 20 LARGEST BUCKETS
block_key
India_priv    46761
US_pedi       31261
India_shri    30313
US_inte       28251
India_limi    19190
US_prim       15866
India_indi    15821
US_metr       15206
India_shiv    14151
US_fami       13720
India_tech    12809
US_card       12416
US_brig       12391
India_shre    11952
US_dent       11472
US_west       11148
US_delt       11139
US_phys       10860
US_corn       10768
US_stra       10746
Name: count, dtype: int64


In [13]:
top_s2_keys = s2["block_key"].value_counts().head(10).index

for key in top_s2_keys:
    print("\n" + "=" * 80)
    print("BLOCK KEY:", key)
    print("COUNT:", (s2["block_key"] == key).sum())

    display(
        s2.loc[
            s2["block_key"] == key,
            ["entity_id", "business_name", "business_address", "country"]
        ].head(20)
    )


BLOCK KEY: India_priv
COUNT: 39818


,entity_id,business_name,business_address,country
41,S2-978126813,Private Ambernath Sólar Limited,"PLOT NO B-78/1, ADDITIONAL MIDC ANAND NAGAR, A...",India
114,S2-435778329,PRIVATE AHMEDABAD INFRATECH (LTD),"A-29 KARAMACHARINAGAR IGHATLODAIA, AHMEDABAD, ...",India
178,S2-308505251,Private Dronagiri College Overseas Limited,"NO.501/C, NULL, BENGALURU, BANGALORE, ಕರ್ನಾಟಕ",India
433,S2-590774517,Private RVA Addibivse Limited,"HINDI SAHIYA PARISHADBUILDING BULANDSAHAR, UTT...",India
458,S2-811707035,Private Sonali Mbamkftisng [Limited],"127, GROUND FLOOR SAHARA SHOPPING CENTRE, INDI...",India
629,S2-369458829,Private New Investments Ltd,"Kerala, KOZHIKODE, 17/581 V",India
641,S2-838493982,Private Seven Global Ltd,"7-10, West Bengal, NADIA, CINEMA HOUSE LANE, K...",India
642,S2-12796093,Private Bombay Klrihnna Systems Limited,"POOVALAMBEDU DIST TIRUVALLUR, 15 OLD NO 8 MADH...",India
643,S2-955730734,Private Shaikpet Capital Ltd,"G-2ND FLOOR, PLOT NO755, ROAD NO36, HYDERABAD,...",India
649,S2-570191750,privateparksquare.com,"3RD FLOOR, RATNANJALI SQUARE, NEAR GLORIYA RES...",India



BLOCK KEY: US_pedi
COUNT: 30732


,entity_id,business_name,business_address,country
61,S2-457500956,Pediatric Dentistry Crystal Care Incorporated,"##6806 ROBIN DR, WY, GILLETTE CITY",US
277,S2-516249560,Pediatric Medicine Partners,"#821 CARRIAGE HOUSE CIRCLE, EVNSVILLE, IN",US
280,S2-716687956,Pediatric Dental Group Inc. Center,"3327 BEDFORD LANE, MONTGOMERY, AL",US
499,S2-716961117,Pediatric Dental Coastal Health Incorporated,"#7528 BOONVILLE HWY, EVANSVILLE, IN",US
510,S2-734993182,Pediatric Dental Partners of Taylorville Inc.,"929 1250 RD N, TAYLORVILLE, IL",US
730,S2-199669656,pediatricdentalspecialists.com,"8636 DEN BALK DRIVE, CHESTERFIELD COUNTY, VA",US
763,S2-176968531,Pediatric Dental Group Inc,"SEEKONK, #23 JUNIPER RD, MA",US
950,S2-310613953,Pediatric Northside Better Physicians [Dental],"3037 IMPRESSIONS DR, LAKE IN THE HILLS, IL",US
1341,S2-92868774,Pediatric Dentistry Associates,"NC, 881 SHORE DR, SOUTHPORT",US
1388,S2-866650875,Pediatric Dentistry Group Group Partners,"12017 OBEE RD, HAVEN, KS",US



BLOCK KEY: US_inte
COUNT: 27623


...

,entity_id,business_name,business_address,country
356,S2-187514897,INDIA CASTINGS COLOUR PRIVATE LTD,"NO 14TH FLOOR, FLAT NO. 1403, A WING DEVCHAYA,...",India
575,S2-781210535,(india) Nanjil Small Pvt,"10, SIKKA COMPLEX, COMMUNITY CENTER, PREET VIH...",India
870,S2-743482966,(india) Hps Hr Co.,"HOUSE NO. 307/1/48, 2ND FLOOR, SHAHZADA BAGH, ...",India
1466,S2-416568090,indigomarketing.com,"Delhi, 2 , 1ST FLOOR, DLF INDUSTRIAL, AREA, NA...",India
2703,S2-678818787,India Classic Services LLP,"HOUSE NO. 589, SECTOR-9 HUDA, AMBALA CITY, Har...",India
3335,S2-489103612,(INDIA) BHAGYASHREE NATURE PRIVATE LIMITED,"Uttar Pradesh, S-05, SECOND FLOOR, OXY HOMEZ, ...",India
3710,S2-981453578,INDIA SUDARSHAN METALS PVT,"SSB 0084, SECTOR 13P, HSAR, हरियाणा",India
4017,S2-335124172,(india) Sarswati India Private Limited,"NO 204 , 2ND FLOOR, GLN GOWDA COMPLEX, BANGLOR...",India
4618,S2-878639783,India India Ss Inpsectin Company,"FLAT - G1702, APARNA SAROVAR ZENITH, NALLAGAND...",India
4651,S2-89956614,Indivar Communications L.L.P.,"#49, PARDA BAGH DARYA GANN, NEW DELHI, Delhi",India



BLOCK KEY: US_fami
COUNT: 13736


,entity_id,business_name,business_address,country
128,S2-857681318,Family Specialists,"4733 LIBERTY RODA, SALEM, OR",US
366,S2-589492301,family care of fairfax county,"4535 WHITTEMORE PLABE, FAIRFAX COUNTY, VA",US
1014,S2-749642407,Family Ridge Chiropractic,"CEDARBROOK ROAD, N/A, NAPERVILLE CITY, IL",US
1300,S2-598991538,Family Medicine (L.L.C.),"04775 WILLIAMS DR, GEORGETOWN, TX",US
1314,S2-626832027,Family Ministries III,"81 MADERA CT, LOS BAONS, CA",US
1389,S2-284585076,FAMILY BLUE CLINIC,"FESTUS, 35 SPRING VALLEY, MO",US
1870,S2-980246492,Family Specialists of Town Of Matteson,"TOWN OF MATTESON, N10958D PEARL FLATS LN, WI",US
2400,S2-980457024,Family Precision Health Corporation,"122, ROUND ROCK, TX",US
2432,S2-520441214,Family Choice Medicine Partners,"IRVINGTON, 40 ECKAR STREET, NY",US
2499,S2-58825780,FAMILY ÍNSTITUTE LLC,"583 NINTH ST, HOUSTON, TX",US



BLOCK KEY: US_card
COUNT: 12399


,entity_id,business_name,business_address,country
202,S2-736654053,Cardiology Clinic Metro Inc.,"256 ROSELAND PLACE, ABERDEEN, NC",US
1258,S2-400132919,CARDIOLOGY CLAOE GROUP,"4950 FULTON DR, FAIRFIELD, CA",US
1354,S2-988228866,Cardiology Bright Care Co,"341 ALLUVIUM DR, LAUREL, MD",US
1369,S2-595414933,CARDIOLOGY CARE ASSOCIATES OF SOMERSET,"4028 Hopewell Twp Rd 40, OH, SOMERSET",US
1993,S2-681384833,cardiologypartners.com,"628 SUN VALLEY DR, ANDERSON, IN",US
2227,S2-97346546,CARDIOLOGY PARTNERS,"550 FLEETWOOD DRIVE, HOT SPRINGS NATIONAL PARK...",US
2976,S2-692356310,Cardiology Care of Des Moines Greater,"##1409 OSCEOLA AVE, DES MOINES, IA",US
3127,S2-49461175,cardiology health inc (ID: 7253),"204 PARKSIDE DRIVE, OH, POWELL",US
3202,S2-846535978,Cardiology Center Pllc,"7837 1700 ROAD NORTH, PONTIAC, IL",US
3447,S2-121549347,Cardiology Associates of Whitman,"66 POND STREET, WHITMAN CITY, MA",US


In [14]:
top_s3_keys = s3["block_key"].value_counts().head(10).index

for key in top_s3_keys:
    print("\n" + "=" * 80)
    print("BLOCK KEY:", key)
    print("COUNT:", (s3["block_key"] == key).sum())

    display(
        s3.loc[
            s3["block_key"] == key,
            ["entity_id", "business_name", "business_address", "country"]
        ].head(20)
    )


BLOCK KEY: India_priv
COUNT: 46761


,entity_id,business_name,business_address,country
68,S3-891901989,Private Shivam Projects [Limited],"B-3 Kailash Colony, New Delhi, South Delhi, DL",India
78,S3-263247365,Private Unique Enterprises Limited,"302, Andheri (West), Mumbai, महाराष्ट्र",India
338,S3-968476116,Private Amika Infotech Límited,B-1/a24- Mohan Co-operativeindustrial Estate M...,India
341,S3-787353706,Private Hitech Infrastructure Ltd,"G-2 Parth 14, Panchnath Plot, Rajkot, ગુજરાત",India
367,S3-126982322,Private Chennai Solutions Limited,"Kanchipuram, TN, No 147, Ground Floor, Nanbarg...",India
388,S3-610402578,PRIVATE PRANAV TECHNOFAB LIMITED,"No.3c, Saravanampatti, Coimbatore, TN",India
461,S3-523695838,Private Sun P0wer Ltd,"Door No 113/1 F-1001, Jewel Of India, Opp. Jai...",India
630,S3-711373924,Private Blue Care Limited,"RJ, 30, Jaipur",India
856,S3-81118620,PRIVATE SELEX CHEMICALS LIMITED,"408, Ahmedabad, GJ",India
860,S3-269732947,Private Lakshmi Global Foods Limited,"Door No 0710 S No 56/5B, Haveli, Pune, MH",India



BLOCK KEY: US_pedi
COUNT: 31261


,entity_id,business_name,business_address,country
138,S3-161412777,pediatric dental care associates inc,"492 Fairborn Ln, Round Lake, Illinois",US
210,S3-903416633,Pediatric Express Care Llc,"6795 Landriano Place, Rancho Cucamonga, Califo...",US
646,S3-538882911,Pediatric Dental Physicians Inc.,NaN,US
1415,S3-697956436,Pediatric Mdeicien of Elkhart West LLC,"0414 Cleveland Ave, Elkhart, Indiana",US
1434,S3-406432606,Pediatric Dental Pcans Inc.,"2623 39th Ave, Washington, Seattle",US
1493,S3-741719955,Pediatric Dental Medicine East Ltd,"18231 34st Avenue, Shaw Butte, Arizona",US
1575,S3-328775191,Pediatric Partners Partners Co,"112 Green St, Boston, Massachusetts",US
1675,S3-39511634,Pediatric Health Partners,"7928 Summerfern Ct, Cypress, Texas",US
1935,S3-775613160,Pediatric Dental Frontier Health Southside [LLC],"9116 Pineville-Matthews Road, # A, Pineville, ...",US
2137,S3-270062152,Pediatric Dentistry Associates Of,"261 River St, Warrensburg, New York",US



BLOCK KEY: India_shri
COUNT: 30313


...

,entity_id,business_name,business_address,country
218,S3-6605683,Metropolitan Holdings Inc,"00524 Marlowe Road, Raleigh, North Carolina",US
394,S3-421946310,Metropolitan Genesis,"20600 Kearney Path, Lakeville, Minnesota",US
560,S3-213092687,Metro Sunstone,"1361 1/2 Firethorne Drive, Mason Ciity, Ohio",US
854,S3-144720166,Metropolitanclassicatlanticus.Com,"620 Washington St, Winchester, Massachusetts",US
1340,S3-76997783,"Metropolitan Hgrózson Steel, LLC","229 Winding Way, <NULL>, Shelbyville, Kentucky",US
1421,S3-303012024,Metropolitan Institute,"03300 Woodlark Dr, Fort Worth, Texas",US
2323,S3-834951367,Metropolitan Horizon Co,"1824 Equine Retre Way, Crandall ICTY, Texas",US
2417,S3-164116745,"Metro Fe1lowship, Group","12232 Schreoder Rd, Dallas, Texas",US
3382,S3-931218676,Metropolitan Federation,"325-327 136th Street, Ocean City, Maryland",US
3521,S3-69542509,"Metro Fróntier United, Inc","1319 Arapahoe Avenue, Salt Lake City, Utah",US



BLOCK KEY: India_shiv
COUNT: 14151


,entity_id,business_name,business_address,country
244,S3-619115665,Shivkripa-Development Private Limited,"&12-7-4/1&2, Hyderabad, TG",India
308,S3-439655728,Shiv Leasing Industries Private,"Birbhum, No 28 C/o, পশ্চিমবঙ্গ",India
592,S3-747355483,shiv business private limited,"Near City Thana Kath Mandi, Divreportingcircle...",India
639,S3-551778816,Shiv Services प्राइवेट लिमिटेड,"Block No. 21, Room No 123, 3Rd Floor, Near Nan...",India
693,S3-895371174,shiva industries private limited,"Sp-g-43, Maurya Enclave Pitampura, New Delhi, DL",India
1908,S3-526379348,SHIVSHAKTI AGENCY PRIVATE LIMITED,"No 120, <NULL>, New Delhi, दिल्ली",India
1973,S3-300764673,Shiv Energy [Limited],"C-20, Ahmedabad, GJ",India
2079,S3-987704104,Shiva Investment Investment L.L.P.,"247, Industrial Area Nangla, Faridabad, HR",India
3226,S3-612463547,Shivam Constructions Pvt Limited,"H.no 68 Ac 45, Palit Villa Prafullkanan, Kesto...",India
4003,S3-93402322,Shiv Consulting,"H.no.3-6-767, Flat.no.302, Anitha Towers Stree...",India



BLOCK KEY: US_fami
COUNT: 13720


,entity_id,business_name,business_address,country
289,S3-400079254,Familycareassociates.Com,"904 Main St, Blackwell, Oklahoma",US
332,S3-971943018,Family East Foundation,"1893 Main St, Newinton, Connecticut",US
981,S3-673495166,Family-Health,"74 Jeter Road, Plao, Illinois",US
2345,S3-87582110,Family LLC Services,"226, Bakersville, North Carolina",US
2452,S3-976237098,Family Bright Care Associates Associates LLC,NaN,US
3020,S3-863259103,#familymountain,"Walgren Dr, Silvedale, Washington",US
3022,S3-853118311,Family-Specialists,"Maryland, Crofton, 1733 Peartree Ln, Fl 0",US
3319,S3-252752998,Family Associates LP,"3415 Coventry Ln, Lafayette, Indiana",US
3886,S3-553176058,Family Medicine East (LLC),"98 Saint Andrews Pl, Unit Apartment 2E, Yonker...",US
4661,S3-513589353,Family Specialists Co,"414- 4nd Street, Gillette, Wyoming",US


In [15]:
print("Ground truth columns:")
print(gt.columns.tolist())

print("\nGround truth shape:")
print(gt.shape)

print("\nGround truth sample:")
display(gt.head())

Ground truth columns:
['source1_entity_id', 'matched_entity_ids']

Ground truth shape:
(2206821, 2)

Ground truth sample:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [16]:
# Create individual S1 -> matched entity pairs
gt_pairs = gt[
    ["source1_entity_id", "matched_entity_ids"]
].copy()

# Convert comma-separated matches into lists
gt_pairs["matched_entity_ids"] = gt_pairs["matched_entity_ids"].fillna("")

gt_pairs["matched_entity_ids"] = gt_pairs["matched_entity_ids"].apply(
    lambda x: [v for v in x.split(",") if v]
)

# One row per true match
gt_pairs = gt_pairs.explode("matched_entity_ids")

# Remove empty matches (singleton/no-match S1 entities)
gt_pairs = gt_pairs[
    gt_pairs["matched_entity_ids"].notna()
    & (gt_pairs["matched_entity_ids"] != "")
].copy()

print("Total ground-truth matched pairs:", len(gt_pairs))
print("\nSample:")
display(gt_pairs.head(10))

Total ground-truth matched pairs: 7638365

Sample:


,source1_entity_id,matched_entity_ids
0,S1-965667,S2-681193310
0,S1-965667,S2-743505751
0,S1-965667,S3-775321672
0,S1-965667,S3-11291185
0,S1-965667,S3-860443364
1,S1-55344266,S2-249013014
1,S1-55344266,S2-197070651
1,S1-55344266,S3-478195123
1,S1-55344266,S3-384364074
2,S1-343815751,S2-790675320


In [17]:
s1_blocks = s1[
    ["entity_id", "block_key"]
].rename(
    columns={
        "entity_id": "source1_entity_id",
        "block_key": "s1_block_key"
    }
)

gt_pairs = gt_pairs.merge(
    s1_blocks,
    on="source1_entity_id",
    how="left"
)

print(gt_pairs.head())

  source1_entity_id matched_entity_ids s1_block_key
0         S1-965667       S2-681193310      US_maur
1         S1-965667       S2-743505751      US_maur
2         S1-965667       S3-775321672      US_maur
3         S1-965667        S3-11291185      US_maur
4         S1-965667       S3-860443364      US_maur


In [19]:
# S2 block-key lookup
s2_blocks = s2[
    ["entity_id", "block_key"]
].rename(
    columns={
        "entity_id": "matched_entity_ids",
        "block_key": "matched_block_key"
    }
)

# S3 block-key lookup
s3_blocks = s3[
    ["entity_id", "block_key"]
].rename(
    columns={
        "entity_id": "matched_entity_ids",
        "block_key": "matched_block_key"
    }
)

print("S2 lookup rows:", len(s2_blocks))
print("S3 lookup rows:", len(s3_blocks))

display(s2_blocks.head())
display(s3_blocks.head())

S2 lookup rows: 5034616
S3 lookup rows: 5285603


,matched_entity_ids,matched_block_key
0,S2-166376419,India_रममर
1,S2-764573417,US_holl
2,S2-639257739,India_आदतय
3,S2-163963287,US_summ
4,S2-49942811,US_delt


,matched_entity_ids,matched_block_key
0,S3-202863386,US_wilf
1,S3-859268022,India_inte
2,S3-22467283,US_llcm
3,S3-671162755,US_moyn
4,S3-960981775,India_pvte


In [20]:
gt_s2 = gt_pairs[
    gt_pairs["matched_entity_ids"].str.startswith("S2-")
].copy()

gt_s3 = gt_pairs[
    gt_pairs["matched_entity_ids"].str.startswith("S3-")
].copy()

print("S2 true pairs:", len(gt_s2))
print("S3 true pairs:", len(gt_s3))
print("Total:", len(gt_s2) + len(gt_s3))

S2 true pairs: 3693619
S3 true pairs: 3944746
Total: 7638365


In [21]:
gt_s2 = gt_s2.merge(
    s2_blocks,
    on="matched_entity_ids",
    how="left"
)

print(gt_s2.shape)
display(gt_s2.head(10))

(3693619, 4)


,source1_entity_id,matched_entity_ids,s1_block_key,matched_block_key
0,S1-965667,S2-681193310,US_maur,US_maur
1,S1-965667,S2-743505751,US_maur,US_maur
2,S1-55344266,S2-249013014,India_raji,India_ரஜஇன
3,S1-55344266,S2-197070651,India_raji,India_raji
4,S1-343815751,S2-790675320,US_dahl,US_dahl
5,S1-343815751,S2-479876582,US_dahl,US_dahl
6,S1-656753428,S2-153058913,India_ssfo,India_एसएस
7,S1-656753428,S2-24659151,India_ssfo,India_एसएस
8,S1-102811957,S2-478959098,US_payn,US_payn
9,S1-102811957,S2-553508714,US_payn,US_payn


In [22]:
gt_s2["captured"] = (
    gt_s2["s1_block_key"]
    == gt_s2["matched_block_key"]
)

print("S2 captured pairs:", gt_s2["captured"].sum())
print("S2 missed pairs:", (~gt_s2["captured"]).sum())

print(
    "S2 blocking recall:",
    f"{gt_s2['captured'].mean():.6%}"
)

S2 captured pairs: 2836235
S2 missed pairs: 857384
S2 blocking recall: 76.787427%


In [23]:
gt_s3 = gt_s3.merge(
    s3_blocks,
    on="matched_entity_ids",
    how="left"
)

gt_s3["captured"] = (
    gt_s3["s1_block_key"]
    == gt_s3["matched_block_key"]
)

print("S3 captured pairs:", gt_s3["captured"].sum())
print("S3 missed pairs:", (~gt_s3["captured"]).sum())

print(
    "S3 blocking recall:",
    f"{gt_s3['captured'].mean():.6%}"
)

S3 captured pairs: 2975792
S3 missed pairs: 968954
S3 blocking recall: 75.436847%


In [24]:
total_pairs = len(gt_s2) + len(gt_s3)

captured_pairs = (
    gt_s2["captured"].sum()
    + gt_s3["captured"].sum()
)

missed_pairs = total_pairs - captured_pairs

overall_recall = captured_pairs / total_pairs

print("=" * 50)
print("V1 BLOCKING RECALL")
print("=" * 50)

print(f"Total true pairs : {total_pairs:,}")
print(f"Captured pairs   : {captured_pairs:,}")
print(f"Missed pairs     : {missed_pairs:,}")
print(f"Recall           : {overall_recall:.6%}")

V1 BLOCKING RECALL
Total true pairs : 7,638,365
Captured pairs   : 5,812,027
Missed pairs     : 1,826,338
Recall           : 76.089935%


In [25]:
missed_s2 = gt_s2[
    ~gt_s2["captured"]
].copy()

missed_s3 = gt_s3[
    ~gt_s3["captured"]
].copy()

print("Missed S2:", len(missed_s2))
print("Missed S3:", len(missed_s3))

Missed S2: 857384
Missed S3: 968954


In [26]:
display(
    missed_s2[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_block_key",
            "matched_block_key"
        ]
    ].head(30)
)

,source1_entity_id,matched_entity_ids,s1_block_key,matched_block_key
2,S1-55344266,S2-249013014,India_raji,India_ரஜஇன
6,S1-656753428,S2-153058913,India_ssfo,India_एसएस
7,S1-656753428,S2-24659151,India_ssfo,India_एसएस
12,S1-318373630,S2-660036492,India_redv,India_रडवच
14,S1-789009573,S2-383871912,India_hote,India_हटलए
21,S1-546142636,S2-392804085,US_crys,US_llcc
27,S1-503957000,S2-994658326,US_apho,US_apin
29,S1-503957000,S2-353308450,US_apho,US_apin
30,S1-503957000,S2-173295926,US_apho,US_apín
32,S1-561341312,S2-483615364,India_bala,India_బలజఇ


In [27]:
display(
    missed_s3[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_block_key",
            "matched_block_key"
        ]
    ].head(30)
)

,source1_entity_id,matched_entity_ids,s1_block_key,matched_block_key
0,S1-965667,S3-775321672,US_maur,US_dréx
4,S1-55344266,S3-384364074,India_raji,India_ரஜஇன
6,S1-656753428,S3-679606215,India_ssfo,India_एसएस
19,S1-730934468,S3-352439310,US_orel,US_llco
20,S1-7293388,S3-523120965,India_chor,India_smtc
21,S1-546142636,S3-200008747,US_crys,US_llcc
22,S1-546142636,S3-729771680,US_crys,US_llcc
26,S1-274126313,S3-850112871,US_obsi,US_korb
27,S1-145361722,S3-96572514,US_dick,US_corp
28,S1-503957000,S3-858763214,US_apho,US_inca


In [28]:
# Number of candidates generated by V1
s1_block_counts = s1["block_key"].value_counts()

v1_candidate_pairs = (
    s1_block_counts
    .to_frame("s1_count")
    .join(
        s2["block_key"].value_counts().rename("s2_count"),
        how="left"
    )
    .join(
        s3["block_key"].value_counts().rename("s3_count"),
        how="left"
    )
    .fillna(0)
)

v1_candidate_pairs["candidate_count"] = (
    v1_candidate_pairs["s1_count"]
    * (
        v1_candidate_pairs["s2_count"]
        + v1_candidate_pairs["s3_count"]
    )
)

print(
    "Total V1 candidate pairs:",
    f"{v1_candidate_pairs['candidate_count'].sum():,}"
)

print(
    "Average candidates per S1:",
    f"{v1_candidate_pairs['candidate_count'].sum() / len(s1):,.2f}"
)

print(
    "Reduction ratio:",
    f"{1 - v1_candidate_pairs['candidate_count'].sum() / (len(s1) * (len(s2)+len(s3))):.4%}"
)

Total V1 candidate pairs: 15,216,021,816.0
Average candidates per S1: 6,895.00
Reduction ratio: 99.9332%


In [29]:
missed = pd.concat(
    [
        missed_s2.assign(target_source="S2"),
        missed_s3.assign(target_source="S3")
    ],
    ignore_index=True
)

print("Total missed:", len(missed))

display(
    missed[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_block_key",
            "matched_block_key",
            "target_source"
        ]
    ].head(50)
)

Total missed: 1826338


,source1_entity_id,matched_entity_ids,s1_block_key,matched_block_key,target_source
0,S1-55344266,S2-249013014,India_raji,India_ரஜஇன,S2
1,S1-656753428,S2-153058913,India_ssfo,India_एसएस,S2
2,S1-656753428,S2-24659151,India_ssfo,India_एसएस,S2
3,S1-318373630,S2-660036492,India_redv,India_रडवच,S2
4,S1-789009573,S2-383871912,India_hote,India_हटलए,S2
5,S1-546142636,S2-392804085,US_crys,US_llcc,S2
6,S1-503957000,S2-994658326,US_apho,US_apin,S2
7,S1-503957000,S2-353308450,US_apho,US_apin,S2
8,S1-503957000,S2-173295926,US_apho,US_apín,S2
9,S1-561341312,S2-483615364,India_bala,India_బలజఇ,S2


In [30]:
s1_info = s1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].rename(
    columns={
        "entity_id": "source1_entity_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

s2_info = s2[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].rename(
    columns={
        "entity_id": "matched_entity_ids",
        "business_name": "matched_name",
        "business_address": "matched_address",
        "country": "matched_country"
    }
)

s3_info = s3[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].rename(
    columns={
        "entity_id": "matched_entity_ids",
        "business_name": "matched_name",
        "business_address": "matched_address",
        "country": "matched_country"
    }
)

In [32]:
# Check what columns are currently available
print(missed_s2.columns.tolist())
print(missed_s2.shape)

['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured']
(857384, 5)


In [35]:
print(
    missed_s2.groupby(
        ["s1_block_key", "matched_block_key"]
    )
    .size()
    .sort_values(ascending=False)
    .head(30)
)

s1_block_key  matched_block_key
India_east    India_ईसटर           2013
India_pion    India_पयनय           1994
India_inno    India_इनवट           1988
India_sout    India_सदरन           1988
India_digi    India_डजटल           1978
India_clas    India_कलसक           1977
India_dyna    India_डयनम           1972
India_adit    India_आदतय           1972
India_arih    India_अरहत           1970
India_futu    India_फयचर           1960
India_laxm    India_लकषम           1958
India_indi    India_इडयन           1944
India_tiru    India_तरपत           1944
India_smar    India_समरट           1943
India_glob    India_गलबल           1937
India_fort    India_फरचय           1929
India_gala    India_गलकस           1927
India_prem    India_परमय           1927
India_mode    India_मडरन           1920
India_silv    India_सलवर           1919
India_sunr    India_सनरइ           1907
India_whit    India_वहइट           1903
India_guja    India_गजरत           1895
India_unit    India_यनइट           1890
India_ap

In [36]:
print(
    missed_s2["s1_block_key"].eq(
        missed_s2["matched_block_key"]
    ).value_counts()
)

False    857384
Name: count, dtype: int64


In [37]:
print(missed_s2["captured"].value_counts(dropna=False))

captured
False    857384
Name: count, dtype: int64


In [38]:
print(
    missed_s2["s1_block_key"]
    .value_counts()
    .head(30)
)

s1_block_key
India_shiv    12145
India_gold     8193
India_east     7958
India_sout     7903
India_shre     5725
India_indi     5146
India_tech     4813
India_glob     4371
India_blue     4359
India_gree     4345
India_kris     4290
India_inte     4277
India_star     4248
India_roya     4228
India_sury     4205
India_bala     4144
India_adit     4132
India_shak     4121
India_high     4116
India_anan     4112
India_real     4084
India_tiru     4083
India_indo     4081
India_guru     4073
India_supe     4073
India_bhar     4051
India_smar     4048
India_futu     4042
India_silv     4029
India_prem     4022
Name: count, dtype: int64


In [39]:
print(
    missed_s2["matched_block_key"]
    .value_counts()
    .head(30)
)

matched_block_key
India_priv    28186
India_shri    18943
India_limi    10159
US_cent        4334
US_serv        4283
India_लकषम     3833
US_llcc        3451
India_smts     3211
India_sris     3167
US_llcs        3036
US_llcp        2731
US_inco        2721
US_corp        2687
US_thec        2679
US_thes        2244
US_llcb        2203
US_llcm        2170
India_ईसटर     2013
India_पयनय     1994
India_सदरन     1988
India_इनवट     1988
India_डजटल     1978
India_कलसक     1977
India_डयनम     1972
India_आदतय     1972
India_अरहत     1970
India_फयचर     1960
India_इडयन     1954
India_हईटक     1953
India_तरपत     1944
Name: count, dtype: int64


In [40]:
%whos DataFrame

Variable             Type         Data/Info
-------------------------------------------
df                   DataFrame    Shape: (2206821, 2)
gt                   DataFrame    Shape: (2206821, 2)
gt_pairs             DataFrame    Shape: (7638365, 3)
gt_s2                DataFrame    Shape: (3693619, 5)
gt_s3                DataFrame    Shape: (3944746, 5)
missed               DataFrame    Shape: (1826338, 6)
missed_s2            DataFrame    Shape: (857384, 5)
missed_s3            DataFrame    Shape: (968954, 5)
s1                   DataFrame    Shape: (2206821, 5)
s1_blocks            DataFrame    Shape: (2206821, 2)
s1_info              DataFrame    Shape: (2206821, 4)
s2                   DataFrame    Shape: (5034616, 5)
s2_blocks            DataFrame    Shape: (5034616, 2)
s2_info              DataFrame    Shape: (5034616, 4)
s3                   DataFrame    Shape: (5285603, 5)
s3_blocks            DataFrame    Shape: (5285603, 2)
s3_info              DataFrame    Shape: (5285603,

In [41]:
print("S1 columns:")
print(s1.columns.tolist())

print("\nS2 columns:")
print(s2.columns.tolist())

print("\nS1 info columns:")
print(s1_info.columns.tolist())

print("\nS2 info columns:")
print(s2_info.columns.tolist())

S1 columns:
['entity_id', 'business_name', 'business_address', 'country', 'block_key']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country', 'block_key']

S1 info columns:
['source1_entity_id', 's1_name', 's1_address', 's1_country']

S2 info columns:
['matched_entity_ids', 'matched_name', 'matched_address', 'matched_country']


In [43]:
display(
    s1.head(3)
)

display(
    s2.head(3)
)

,entity_id,business_name,business_address,country,block_key
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,US_orel
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,US_prim
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,US_bret


,entity_id,business_name,business_address,country,block_key
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,India_रममर
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,US_holl
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,India_आदतय


In [44]:
# Attach actual S1 and S2 records to missed S2 pairs

missed_s2_detail = (
    missed_s2[
        ["source1_entity_id", "matched_entity_ids",
         "s1_block_key", "matched_block_key"]
    ]
    .merge(
        s1[
            ["entity_id", "business_name",
             "business_address", "country"]
        ],
        left_on="source1_entity_id",
        right_on="entity_id",
        how="left"
    )
    .drop(columns=["entity_id"])
    .rename(columns={
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    })
    .merge(
        s2[
            ["entity_id", "business_name",
             "business_address", "country"]
        ],
        left_on="matched_entity_ids",
        right_on="entity_id",
        how="left"
    )
    .drop(columns=["entity_id"])
    .rename(columns={
        "business_name": "s2_name",
        "business_address": "s2_address",
        "country": "s2_country"
    })
)

print(missed_s2_detail.shape)

display(
    missed_s2_detail[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "s2_name",
            "s1_address",
            "s2_address",
            "s1_country",
            "s2_country",
            "s1_block_key",
            "matched_block_key"
        ]
    ].head(30)
)

(857384, 10)


,source1_entity_id,matched_entity_ids,s1_name,s2_name,s1_address,s2_address,s1_country,s2_country,s1_block_key,matched_block_key
0,S1-55344266,S2-249013014,Raj Investments LLP,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...","6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, ...",India,India,India_raji,India_ரஜஇன
1,S1-656753428,S2-153058913,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,"Af-684, Nandgram Near Mother India Public Scho...","AF-0684, NANDGRAM NEAR MOTHER INDIA PUBLIC SCH...",India,India,India_ssfo,India_एसएस
2,S1-656753428,S2-24659151,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,"Af-684, Nandgram Near Mother India Public Scho...","AF-0684, Uttar Pradesh, GHAZIABAD, 9487203",India,India,India_ssfo,India_एसएस
3,S1-318373630,S2-660036492,Red Ventures Private Limited,रेड वेंचर्स प्राइवेट लिमिटेड,"Rajasthan, Jaipur, Banipark, Gokul Apartment, ...","G-1, BANIPARK, JAIPUR, Rajasthan",India,India,India_redv,India_रडवच
4,S1-789009573,S2-383871912,Hotel Enterprises Limited,होटल एंटरप्राइजेज लिमिटेड,"Wz-187C Shop No.13, 14 Kh. No.47 S/F. Vikaspur...","WZ-187C SHOP NO.13, DELHI, WEST DELHI, Delhi",India,India,India_hote,India_हटलए
5,S1-546142636,S2-392804085,Crystal Staffing Solutions LLC,LLC Crystal Sttfrifng Solutions,"8706 Kentucky Derby Drive, Waxhaw, NC","8706 KENTUCKY DERBY DRIVE, WAXHAW, NC",US,US,US_crys,US_llcc
6,S1-503957000,S2-994658326,AP Hospitality Inc,AP INC SERVICE #80430,"20085 Us 23, Circleville, OH","20085 US 23, CIRCLEVILLE, OH",US,US,US_apho,US_apin
7,S1-503957000,S2-353308450,AP Hospitality Inc,AP Inc Hospitality,"20085 Us 23, Circleville, OH","20085 US 23, CIRCLEVILLE, OH",US,US,US_apho,US_apin
8,S1-503957000,S2-173295926,AP Hospitality Inc,AP Ínc Center,"20085 Us 23, Circleville, OH","20085 US 23, CIRCLEVILLE, OH",US,US,US_apho,US_apín
9,S1-561341312,S2-483615364,Balaji Investment Private Limited,బాలాజీ ఇన్వెస్ట్‌మెంట్ ప్రైవేట్ లిమిటెడ్,"Plot No. D-88 & D-90, Hyd, Telangana, Hyderaba...","H.NO 00516 PLOT NO. D-88 & D-90, JEEDIMETLA, H...",India,India,India_bala,India_బలజఇ


In [45]:
import re
import pandas as pd

def extract_postal(address):
    if pd.isna(address):
        return None
    
    address = str(address)

    # Indian PIN: exactly 6 digits
    m = re.search(r'(?<!\d)(\d{6})(?!\d)', address)
    if m:
        return m.group(1)

    # US ZIP: 5 digits or ZIP+4
    m = re.search(r'(?<!\d)(\d{5})(?:-\d{4})?(?!\d)', address)
    if m:
        return m.group(1)

    return None

In [46]:
s1["postal_key"] = s1["business_address"].apply(extract_postal)
s2["postal_key"] = s2["business_address"].apply(extract_postal)

print("S1 postal coverage:",
      s1["postal_key"].notna().mean())

print("S2 postal coverage:",
      s2["postal_key"].notna().mean())

S1 postal coverage: 0.06768559842415856
S2 postal coverage: 0.0755743834286468


In [47]:
s1_postal = s1[
    ["entity_id", "postal_key"]
].rename(
    columns={"entity_id": "source1_entity_id"}
)

s2_postal = s2[
    ["entity_id", "postal_key"]
].rename(
    columns={"entity_id": "matched_entity_ids"}
)

In [48]:
gt_postal = (
    gt_pairs
    .merge(
        s1_postal,
        on="source1_entity_id",
        how="left"
    )
    .merge(
        s2_postal,
        on="matched_entity_ids",
        how="left",
        suffixes=("_s1", "_s2")
    )
)

postal_captured = (
    gt_postal["postal_key_s1"].notna()
    &
    gt_postal["postal_key_s2"].notna()
    &
    (gt_postal["postal_key_s1"] == gt_postal["postal_key_s2"])
)

print("True pairs:", len(gt_postal))
print("Postal captured:", postal_captured.sum())
print("Postal recall:",
      postal_captured.mean() * 100)

True pairs: 7638365
Postal captured: 182275
Postal recall: 2.386309111963097


In [49]:
s1["postal_block"] = (
    s1["country"].astype(str)
    + "_"
    + s1["postal_key"].fillna("")
)

s2["postal_block"] = (
    s2["country"].astype(str)
    + "_"
    + s2["postal_key"].fillna("")
)

In [51]:
print(
    s1[s1["postal_key"].notna()]
    .groupby("postal_block")
    .size()
    .describe()
)

print(
    s2[s2["postal_key"].notna()]
    .groupby("postal_block")
    .size()
    .describe()
)

count    34472.000000
mean         4.333082
std          6.345921
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max         87.000000
dtype: float64
count    59981.000000
mean         6.343475
std          9.913418
min          1.000000
25%          1.000000
50%          3.000000
75%          7.000000
max        207.000000
dtype: float64


In [52]:
missed_postal = (
    missed_s2
    .merge(
        s1[
            ["entity_id", "postal_block"]
        ],
        left_on="source1_entity_id",
        right_on="entity_id",
        how="left"
    )
    .drop(columns=["entity_id"])
    .merge(
        s2[
            ["entity_id", "postal_block"]
        ],
        left_on="matched_entity_ids",
        right_on="entity_id",
        how="left",
        suffixes=("_s1", "_s2")
    )
    .drop(columns=["entity_id"])
)

postal_recovers_missed = (
    missed_postal["postal_block_s1"].notna()
    &
    missed_postal["postal_block_s2"].notna()
    &
    (
        missed_postal["postal_block_s1"]
        ==
        missed_postal["postal_block_s2"]
    )
)

print("Missed S2:", len(missed_s2))
print(
    "Recovered by postal:",
    postal_recovers_missed.sum()
)
print(
    "Recovery:",
    postal_recovers_missed.mean() * 100,
    "%"
)

Missed S2: 857384
Recovered by postal: 837176
Recovery: 97.64306308491878 %


In [54]:
print("Original S2 missed:", len(missed_s2))
print("Recovered by postal:", 837176)
print("Remaining S2 missed:", len(missed_s2) - 837176)

Original S2 missed: 857384
Recovered by postal: 837176
Remaining S2 missed: 20208


In [55]:
missed_s2_remaining = missed_s2[missed_s2["captured"] == False].copy()

print("Remaining S2 missed:", len(missed_s2_remaining))

Remaining S2 missed: 857384


In [61]:
import pandas as pd

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if {
            "source1_entity_id",
            "matched_entity_ids"
        }.issubset(cols):

            if len(obj) == 837176:
                print(
                    "FOUND:",
                    name,
                    "| shape:",
                    obj.shape,
                    "| columns:",
                    list(obj.columns)
                )

In [63]:
import pandas as pd

print("DataFrames containing S2-related information:\n")

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):

        cols = list(obj.columns)

        relevant = any(
            c in cols
            for c in [
                "source1_entity_id",
                "matched_entity_ids",
                "s1_entity_id",
                "s2_entity_id",
                "s1_postal",
                "s2_postal",
                "postal",
                "captured"
            ]
        )

        if relevant:
            print(
                f"{name:30} "
                f"shape={obj.shape} "
                f"columns={cols}"
            )

DataFrames containing S2-related information:

gt                             shape=(2206821, 2) columns=['source1_entity_id', 'matched_entity_ids']
df                             shape=(2206821, 2) columns=['source1_entity_id', 'matched_entity_ids']
gt_pairs                       shape=(7638365, 3) columns=['source1_entity_id', 'matched_entity_ids', 's1_block_key']
s1_blocks                      shape=(2206821, 2) columns=['source1_entity_id', 's1_block_key']
s2_blocks                      shape=(5034616, 2) columns=['matched_entity_ids', 'matched_block_key']
s3_blocks                      shape=(5285603, 2) columns=['matched_entity_ids', 'matched_block_key']
gt_s2                          shape=(3693619, 5) columns=['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured']
gt_s3                          shape=(3944746, 5) columns=['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured']
missed_s2                    

In [65]:
print("\nVariables containing 'postal', 's2', or 'miss':\n")

for name in globals():
    name_lower = name.lower()

    if (
        "postal" in name_lower
        or "s2" in name_lower
        or "miss" in name_lower
    ):
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            print(f"{name:30} DataFrame {obj.shape}")
        else:
            print(f"{name:30} {type(obj).__name__}")


Variables containing 'postal', 's2', or 'miss':

s2                             DataFrame (5034616, 7)
top_s2_keys                    Index
s2_blocks                      DataFrame (5034616, 2)
gt_s2                          DataFrame (3693619, 5)
missed_pairs                   int64
missed_s2                      DataFrame (857384, 5)
missed_s3                      DataFrame (968954, 5)
missed                         DataFrame (1826338, 6)
s2_info                        DataFrame (5034616, 4)
missed_s2_detail               DataFrame (857384, 10)
extract_postal                 function
s1_postal                      DataFrame (2206821, 2)
s2_postal                      DataFrame (5034616, 2)
gt_postal                      DataFrame (7638365, 5)
postal_captured                Series
missed_postal                  DataFrame (857384, 7)
postal_recovers_missed         Series
missed_s2_remaining            DataFrame (857384, 6)


In [66]:
# Make sure the postal recovery mask aligns with missed_s2
postal_recovers_missed = postal_recovers_missed.reindex(
    missed_s2.index,
    fill_value=False
)

# Keep only S2 pairs NOT recovered by postal blocking
missed_s2_remaining = missed_s2[
    ~postal_recovers_missed
].copy()

print("Original S2 missed:", len(missed_s2))
print("Recovered by postal:", postal_recovers_missed.sum())
print("Remaining S2 missed:", len(missed_s2_remaining))

Original S2 missed: 857384
Recovered by postal: 194364
Remaining S2 missed: 663020


In [67]:
print("Expected remaining:", 857384 - 837176)
print("Actual remaining:", len(missed_s2_remaining))

print("\nRecovery percentage:")
print(
    round(
        postal_recovers_missed.sum() / len(missed_s2) * 100,
        4
    ),
    "%"
)

Expected remaining: 20208
Actual remaining: 663020

Recovery percentage:
22.6694 %


In [69]:
print("missed_s2:", missed_s2.shape)
print("missed_postal:", missed_postal.shape)

print("\npostal_recovers_missed:")
print("length:", len(postal_recovers_missed))
print("index matches missed_s2:", postal_recovers_missed.index.equals(missed_s2.index))
print("True:", postal_recovers_missed.sum())
print("False:", (~postal_recovers_missed).sum())

print("\nmissed_postal columns:")
print(missed_postal.columns.tolist())

print("\nPostal block equality:")
postal_equal = (
    missed_postal["postal_block_s1"].notna()
    & missed_postal["postal_block_s2"].notna()
    & (
        missed_postal["postal_block_s1"]
        == missed_postal["postal_block_s2"]
    )
)

print("Equal postal blocks:", postal_equal.sum())
print("Not equal:", (~postal_equal).sum())

print("\nSample:")
display(
    missed_postal[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "postal_block_s1",
            "postal_block_s2"
        ]
    ].head(20)
)

missed_s2: (857384, 5)
missed_postal: (857384, 7)

postal_recovers_missed:
length: 857384
index matches missed_s2: True
True: 194364
False: 663020

missed_postal columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 'postal_block_s1', 'postal_block_s2']

Postal block equality:
Equal postal blocks: 837176
Not equal: 20208

Sample:


,source1_entity_id,matched_entity_ids,postal_block_s1,postal_block_s2
0,S1-55344266,S2-249013014,India_,India_
1,S1-656753428,S2-153058913,India_,India_
2,S1-656753428,S2-24659151,India_,India_
3,S1-318373630,S2-660036492,India_,India_
4,S1-789009573,S2-383871912,India_,India_
5,S1-546142636,S2-392804085,US_,US_
6,S1-503957000,S2-994658326,US_20085,US_20085
7,S1-503957000,S2-353308450,US_20085,US_20085
8,S1-503957000,S2-173295926,US_20085,US_20085
9,S1-561341312,S2-483615364,India_,India_00516


In [72]:
# Valid postal match:
# 1. Both postal keys exist
# 2. They are equal
# 3. The key contains an actual postal value after "India_" / "US_"

postal_s1 = missed_postal["postal_block_s1"].fillna("").astype(str)
postal_s2 = missed_postal["postal_block_s2"].fillna("").astype(str)

valid_postal_recovery = (
    postal_s1.ne("")
    & postal_s2.ne("")
    & postal_s1.eq(postal_s2)
    & ~postal_s1.isin(["India_", "US_"])
)

print("Valid postal recoveries:", valid_postal_recovery.sum())
print("Invalid/remaining:", (~valid_postal_recovery).sum())

Valid postal recoveries: 23756
Invalid/remaining: 833628


In [73]:
empty_postal_match = (
    postal_s1.eq(postal_s2)
    & postal_s1.isin(["India_", "US_"])
)

print("Total equal postal keys:", postal_s1.eq(postal_s2).sum())
print("Empty postal matches:", empty_postal_match.sum())
print(
    "Actual postal matches:",
    postal_s1.eq(postal_s2).sum() - empty_postal_match.sum()
)

Total equal postal keys: 837176
Empty postal matches: 813420
Actual postal matches: 23756


In [81]:
# ============================================================
# STEP 4: VALID POSTAL RECOVERIES
# ============================================================

# Make sure both columns are strings
postal_s1 = missed_postal["postal_block_s1"].fillna("").astype(str)
postal_s2 = missed_postal["postal_block_s2"].fillna("").astype(str)

# Valid postal match:
# - both sides equal
# - not empty
# - not just "India_" or "US_"
valid_postal_mask = (
    postal_s1.eq(postal_s2)
    & postal_s1.ne("")
    & ~postal_s1.isin(["India_", "US_"])
)

print("Valid postal recoveries:", valid_postal_mask.sum())
print("Invalid/remaining:", (~valid_postal_mask).sum())

Valid postal recoveries: 23756
Invalid/remaining: 833628


In [85]:
# ============================================================
# STEP 5A — Build detailed remaining S2 dataframe
# ============================================================

missed_s2_remaining_detail = (
    missed_s2_remaining
    .merge(
        s1_info,
        on="source1_entity_id",
        how="left"
    )
    .merge(
        s2_info,
        on="matched_entity_ids",
        how="left"
    )
)

print("Shape:", missed_s2_remaining_detail.shape)
print("\nColumns:")
print(missed_s2_remaining_detail.columns.tolist())

Shape: (663020, 11)

Columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country']


In [86]:
display(
    missed_s2_remaining_detail[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "s1_country",
            "matched_country",
            "s1_block_key",
            "matched_block_key"
        ]
    ].head(30)
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,s1_address,matched_address,s1_country,matched_country,s1_block_key,matched_block_key
0,S1-307962633,S2-415653047,Historical Committee Inc.,HÍSTORICALCOMMITTEE.COM,"3823 Hidden Cove Court, Rockwall, TX","HIDDEN COVE CT, PMB 4364, ROCKWWALL, TX",US,US,US_hist,US_híst
1,S1-806616139,S2-512833764,Golden Power Pvt Ltd,gpower.com,"431, 441 F/F Front Side, Gali No.5 Guru Ram Da...","431, दिल्ली, EAST DELHI, 441 F/F FRONT SIDE, G...",India,India,India_gold,India_gpow
2,S1-740600107,S2-303262676,Tech Enterprises (India) Limited,Shri TECH ENTERPRISES LIMITED CENTER,"53/4, Nehru Nagar West, Coimbatore South, Coim...","53/4, NEHRU NAGAR WESM, COIMBATORE SOUTH, Tami...",India,India,India_tech,India_shri
3,S1-854707534,S2-19061664,Peridyn Chain LLC,LLC Peridyn Cháin,"NY, Brookhaven, 6 Heron Path","6-D HERON PATH, CORAM, NY",US,US,US_peri,US_llcp
4,S1-291419333,S2-352538774,Cozy Hypnosis!,C0zy Hypnósis!,"121 Friendship Street, Unit 2, Fall River, MA","FAL LRIVER, 121 FRIENDSHIP STREET, MA",US,US,US_cozy,US_c0zy
5,S1-602144510,S2-32696802,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delh...","C-93, DELHI, NEW DELHI, Delhi",India,India,India_inte,India_इटरन
6,S1-602144510,S2-64000798,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delh...","C-93, G/F, RING ROAD, NDSE-II, DELHI, NEW DELH...",India,India,India_inte,India_इटरन
7,S1-559249977,S2-312080751,My Investment Private Limited,माय इन्वेस्टमेंट प्राइवेट लिमिटेड,"C/O Mayank Anilkumar Sing, Chinchani District ...","C/O MAYANK ANILKUMAR SING, PALGHAR, THANE, Mah...",India,India,India_myin,India_मयइन
8,S1-507257254,S2-420623386,Galaxy Consultancy Limited,ಗ್ಯಾಲಕ್ಸಿ ಕನ್ಸಲ್ಟೆನ್ಸಿ ಲಿಮಿಟೆಡ್,"#270 10Th Main 2Nd Block Jayanagar, Bangalore,...","BENGALURU, #G-270 10TH MAIN 2ND BLOCK JAYANAGA...",India,India,India_gala,India_ಗಯಲಕ
9,S1-715642052,S2-484970278,Ds Properties Private Limited,Ds Limited Private Properties Properties,"Ambika Kunj, 194D, Satin Sen Sarani, 3Rd Floor...","পশ্চিমবঙ্গ, DOOR NO 112 AMBIKA KUNJ, 194D, SAT...",India,India,India_dspr,India_dsli


In [87]:
display(
    missed_s2_remaining_detail[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "s1_country",
            "matched_country",
            "s1_block_key",
            "matched_block_key"
        ]
    ].head(30)
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,s1_address,matched_address,s1_country,matched_country,s1_block_key,matched_block_key
0,S1-307962633,S2-415653047,Historical Committee Inc.,HÍSTORICALCOMMITTEE.COM,"3823 Hidden Cove Court, Rockwall, TX","HIDDEN COVE CT, PMB 4364, ROCKWWALL, TX",US,US,US_hist,US_híst
1,S1-806616139,S2-512833764,Golden Power Pvt Ltd,gpower.com,"431, 441 F/F Front Side, Gali No.5 Guru Ram Da...","431, दिल्ली, EAST DELHI, 441 F/F FRONT SIDE, G...",India,India,India_gold,India_gpow
2,S1-740600107,S2-303262676,Tech Enterprises (India) Limited,Shri TECH ENTERPRISES LIMITED CENTER,"53/4, Nehru Nagar West, Coimbatore South, Coim...","53/4, NEHRU NAGAR WESM, COIMBATORE SOUTH, Tami...",India,India,India_tech,India_shri
3,S1-854707534,S2-19061664,Peridyn Chain LLC,LLC Peridyn Cháin,"NY, Brookhaven, 6 Heron Path","6-D HERON PATH, CORAM, NY",US,US,US_peri,US_llcp
4,S1-291419333,S2-352538774,Cozy Hypnosis!,C0zy Hypnósis!,"121 Friendship Street, Unit 2, Fall River, MA","FAL LRIVER, 121 FRIENDSHIP STREET, MA",US,US,US_cozy,US_c0zy
5,S1-602144510,S2-32696802,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delh...","C-93, DELHI, NEW DELHI, Delhi",India,India,India_inte,India_इटरन
6,S1-602144510,S2-64000798,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delh...","C-93, G/F, RING ROAD, NDSE-II, DELHI, NEW DELH...",India,India,India_inte,India_इटरन
7,S1-559249977,S2-312080751,My Investment Private Limited,माय इन्वेस्टमेंट प्राइवेट लिमिटेड,"C/O Mayank Anilkumar Sing, Chinchani District ...","C/O MAYANK ANILKUMAR SING, PALGHAR, THANE, Mah...",India,India,India_myin,India_मयइन
8,S1-507257254,S2-420623386,Galaxy Consultancy Limited,ಗ್ಯಾಲಕ್ಸಿ ಕನ್ಸಲ್ಟೆನ್ಸಿ ಲಿಮಿಟೆಡ್,"#270 10Th Main 2Nd Block Jayanagar, Bangalore,...","BENGALURU, #G-270 10TH MAIN 2ND BLOCK JAYANAGA...",India,India,India_gala,India_ಗಯಲಕ
9,S1-715642052,S2-484970278,Ds Properties Private Limited,Ds Limited Private Properties Properties,"Ambika Kunj, 194D, Satin Sen Sarani, 3Rd Floor...","পশ্চিমবঙ্গ, DOOR NO 112 AMBIKA KUNJ, 194D, SAT...",India,India,India_dspr,India_dsli


In [88]:
print(
    missed_s2_remaining_detail[
        ["s1_country", "matched_country"]
    ].value_counts()
)

s1_country  matched_country
India       India              459336
US          US                 203684
Name: count, dtype: int64


In [89]:
block_mismatch = (
    missed_s2_remaining_detail
    .groupby(
        ["s1_block_key", "matched_block_key"]
    )
    .size()
    .sort_values(ascending=False)
)

display(block_mismatch.head(50))

s1_block_key  matched_block_key
India_pion    India_पयनय           1605
India_east    India_ईसटर           1560
India_laxm    India_लकषम           1547
India_sout    India_सदरन           1542
India_inno    India_इनवट           1540
India_digi    India_डजटल           1533
India_tiru    India_तरपत           1526
India_fort    India_फरचय           1521
India_arih    India_अरहत           1518
India_clas    India_कलसक           1517
India_silv    India_सलवर           1506
India_guja    India_गजरत           1501
India_futu    India_फयचर           1497
India_adit    India_आदतय           1495
India_glob    India_गलबल           1494
India_sunr    India_सनरइ           1487
India_mode    India_मडरन           1487
India_indi    India_इडयन           1482
India_prem    India_परमय           1479
India_apex    India_एपकस           1471
India_whit    India_वहइट           1470
India_unit    India_यनइट           1469
India_gala    India_गलकस           1467
India_swas    India_सवसत           1466
India_dy

In [90]:
same_country = (
    missed_s2_remaining_detail["s1_country"]
    .astype(str)
    .str.lower()
    ==
    missed_s2_remaining_detail["matched_country"]
    .astype(str)
    .str.lower()
)

print("Same country:", same_country.sum())
print("Different country:", (~same_country).sum())
print("Same-country %:", same_country.mean() * 100)

Same country: 663020
Different country: 0
Same-country %: 100.0


In [95]:
import sys
!{sys.executable} -m pip install rapidfuzz

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 11.8 MB/s  0:00:00


In [96]:
import sys
print(sys.executable)

!{sys.executable} -m pip show rapidfuzz

c:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe
Name: RapidFuzz
Version: 3.14.6
Summary: rapid fuzzy string matching
Home-page: https://github.com/rapidfuzz/RapidFuzz
Author: 
Author-email: Max Bachmann <pypi@maxbachmann.de>
License-Expression: MIT
Location: c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: 
Required-by: 


In [97]:
from rapidfuzz.fuzz import ratio

print(ratio("hello", "helo"))

88.88888888888889


In [98]:
import sys
print(sys.executable)

c:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe


In [100]:
# Calculate name and address similarity
remaining = missed_s2_remaining_detail.copy()

remaining["name_s1_norm"] = remaining["s1_name"].map(normalize_text)
remaining["name_s2_norm"] = remaining["matched_name"].map(normalize_text)

remaining["address_s1_norm"] = remaining["s1_address"].map(normalize_text)
remaining["address_s2_norm"] = remaining["matched_address"].map(normalize_text)

remaining["name_score"] = [
    ratio(a, b)
    for a, b in zip(
        remaining["name_s1_norm"],
        remaining["name_s2_norm"]
    )
]

remaining["address_score"] = [
    ratio(a, b)
    for a, b in zip(
        remaining["address_s1_norm"],
        remaining["address_s2_norm"]
    )
]

remaining["combined_score"] = (
    0.65 * remaining["name_score"] +
    0.35 * remaining["address_score"]
)

print("Remaining candidates:", len(remaining))

print("\nScore distribution:")
display(
    remaining[
        ["name_score", "address_score", "combined_score"]
    ].describe()
)

Remaining candidates: 663020

Score distribution:


,name_score,address_score,combined_score
count,663020.000000,663020.000000,663020.000000
mean,46.723148,78.050844,57.687841
std,33.081084,20.378957,23.406681
min,0.000000,0.000000,8.666667
25%,10.526316,65.486726,36.468558
50%,54.054054,83.870968,60.443350
75%,78.260870,94.117647,79.402768
max,98.630137,100.000000,98.773585


In [101]:
display(
    remaining.sort_values(
        "combined_score",
        ascending=False
    )[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ].head(100)
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,s1_address,matched_address,name_score,address_score,combined_score
323891,S1-917424756,S2-904051473,Michael Holdings Group LLC,Mnichael Holdings Group LLC,"2950 Arizona Way, Manchester, MD","#2950 ARIZONA WAY, MANCHESTER, MD",98.113208,100.0,98.773585
172544,S1-633447588,S2-971801269,Al Skilled Private Limited,Al Sakilled Private Limited,"Chinchani, Tal Tasgaon Dist Sangli, Sangli, Ma...","CHINCHANI, TAL TASGAON DIST SANGLI, SANGLI, Ma...",98.113208,100.0,98.773585
619790,S1-33212034,S2-446786670,CC Global Private Limited,CC Gnlobal Private Limited,"Flat No. A 501 Bcm Height, Pu-4 Scheme N0. 54,...","FLAT NO. A 501 BCM HEIGHT, PU-4 SCHEME N0. 54...",98.039216,100.0,98.725490
308687,S1-795310232,S2-976744875,AN Herbal Private Limited,AN Hgerbal Private Limited,"305 P H Complex Vijay Block, Laxmi Nagar, New ...","305 P H COMPLEX VIJAY BLOCK, LAXMI NAGAR, NEW ...",98.039216,100.0,98.725490
87716,S1-479403742,S2-64080606,Wareham Holdings Group LLC,Warham Holdings Group [LLC],"1013 Fargo Drive, Mckinney, TX","1013 FARGO DRIVE, MCKINNEY, TX",98.039216,100.0,98.725490
...,...,...,...,...,...,...,...,...,...
296535,S1-127580558,S2-847400591,Philippe Murphy Custom Frequency LLC,Phi1ippe Murphy Custom Frequency LLC,"92-62 Queens Boulevard, Rego Park, NY","92-62 QUEENS BOULEVARD, REGO PARK, NY",97.222222,100.0,98.194444
277188,S1-165700724,S2-934779565,Corporate Education Laboratories LLC,C0rporate Education Laboratories LLC,"55 Burton Road, Franklin, WV","55 BURTON ROAD, FRANKLIN, WV",97.222222,100.0,98.194444
126129,S1-395267752,S2-712084495,"Joletta Carlson Unified Vertical, Inc.","J0letta Carlson Unified Vertical, Inc.","38 Harbor Heights Road, Scituate, MA","38 Harbor Heights Road, SCITUATE, MA",97.222222,100.0,98.194444
11591,S1-685829932,S2-577038988,Usg (India) Infraconstructions Pvt Ltd,Usg (Índia) Infraconstructions Pvt Ltd,Flat No A/602 Rameshwar Neelkanth Heights Pokh...,FLAT NO A/602 RAMESHWAR NEELKANTH HEIGHTS POKH...,97.222222,100.0,98.194444


In [102]:
# Build a sample of known TRUE S2 pairs
true_s2 = gt_s2.copy()

# Attach S1 information
true_s2 = true_s2.merge(
    s1_info,
    on="source1_entity_id",
    how="left"
)

# Attach S2 information
true_s2 = true_s2.merge(
    s2_info,
    on="matched_entity_ids",
    how="left"
)

print("True S2 pairs:", len(true_s2))
print(true_s2.columns.tolist())

True S2 pairs: 3693619
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country']


In [103]:
true_s2["name_s1_norm"] = true_s2["s1_name"].map(normalize_text)
true_s2["name_s2_norm"] = true_s2["matched_name"].map(normalize_text)

true_s2["address_s1_norm"] = true_s2["s1_address"].map(normalize_text)
true_s2["address_s2_norm"] = true_s2["matched_address"].map(normalize_text)

true_s2["name_score"] = [
    ratio(a, b)
    for a, b in zip(
        true_s2["name_s1_norm"],
        true_s2["name_s2_norm"]
    )
]

true_s2["address_score"] = [
    ratio(a, b)
    for a, b in zip(
        true_s2["address_s1_norm"],
        true_s2["address_s2_norm"]
    )
]

true_s2["combined_score"] = (
    0.65 * true_s2["name_score"] +
    0.35 * true_s2["address_score"]
)

display(
    true_s2[
        ["name_score", "address_score", "combined_score"]
    ].describe()
)

,name_score,address_score,combined_score
count,3.693619e+06,3.693619e+06,3.693619e+06
mean,7.877297e+01,7.909784e+01,7.888667e+01
std,2.586183e+01,2.305207e+01,1.896701e+01
min,0.000000e+00,0.000000e+00,8.125000e+00
25%,7.272727e+01,7.111111e+01,7.282430e+01
50%,8.780488e+01,8.656716e+01,8.471272e+01
75%,9.615385e+01,9.473684e+01,9.225191e+01
max,1.000000e+02,1.000000e+02,1.000000e+02


In [104]:
display(
    true_s2.sort_values(
        "combined_score",
        ascending=True
    )[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ].head(100)
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,s1_address,matched_address,name_score,address_score,combined_score
135233,S1-379082029,S2-980240110,# 4 NA Coinshares,# 4,"1477 Grayson Street, Nocona, TX",NaN,12.500000,0.000000,8.125000
2819644,S1-994168313,S2-601773616,D & N Clough LLC,#d,"Hebron, 3131 30, NY",NaN,13.333333,0.000000,8.666667
3300460,S1-6684419,S2-260842643,# 3 SE Research,# 3,"357 Delgany Trail, Haslet, TX",NaN,14.285714,0.000000,9.285714
1022891,S1-24835166,S2-928335426,P & L Lafayette,P (&),"4816 Ancestry Street, Haltom City, TX",NaN,14.285714,0.000000,9.285714
1308780,S1-313042577,S2-638425682,F & U Altenergy,F-&,"TN, 1448 Cottage Glen Lane, Maryville",NaN,14.285714,0.000000,9.285714
...,...,...,...,...,...,...,...,...,...
37841,S1-42151717,S2-71467835,Sai Infrastructure Private Limited,साई इंफ्रास्ट्रक्चर प्राइवेट लिमिटेड,"Fl C, 1401, M Sb, K S Galaxy, Plot 11 And 15 S...","RAIGARH, FL C, महाराष्ट्र",8.571429,19.298246,12.325815
2281911,S1-958515461,S2-826643271,Sunrise Foundation,सनराइज फाउंडेशन,"216, Kohinoor Arcade, 2Nd Floor Tilak Chowk, M...","216, PUNE, महाराष्ट्र",6.060606,24.000000,12.339394
179233,S1-861331864,S2-725468780,Global Construction,ગ્લોબલ કન્સ્ટ્રક્શન,"406 4Th Floor, W-1, Opp Psp House, B/H S. G. H...","ગુજરાત, AHMEDABD, ##406 4TH FLOOR",5.263158,25.490196,12.342621
2749513,S1-604801234,S2-151440931,Green Foundation,ಗ್ರೀನ್ ಫೌಂಡೇಶನ್,"Plot No. 21, 32, 33, 34, Kiadb Industrial Area...","316, MANDYA, ಕರ್ನಾಟಕ",7.142857,22.000000,12.342857


In [107]:
thresholds = [80, 85, 90, 92, 94, 95, 96, 97, 98]

print("REMAINING CANDIDATES")
print("=" * 70)

for t in thresholds:
    n = (remaining["combined_score"] >= t).sum()
    pct = n / len(remaining) * 100

    print(
        f"Threshold >= {t}: "
        f"{n:,} candidates ({pct:.2f}%)"
    )

print("\nTRUE S2 PAIRS")
print("=" * 70)

for t in thresholds:
    n = (true_s2_scored["combined_score"] >= t).sum()
    pct = n / len(true_s2_scored) * 100

    print(
        f"Threshold >= {t}: "
        f"{n:,} true pairs ({pct:.2f}% recall)"
    )

REMAINING CANDIDATES
Threshold >= 80: 158,897 candidates (23.97%)
Threshold >= 85: 99,629 candidates (15.03%)
Threshold >= 90: 45,486 candidates (6.86%)
Threshold >= 92: 27,402 candidates (4.13%)
Threshold >= 94: 13,427 candidates (2.03%)
Threshold >= 95: 8,194 candidates (1.24%)
Threshold >= 96: 3,911 candidates (0.59%)
Threshold >= 97: 1,474 candidates (0.22%)
Threshold >= 98: 197 candidates (0.03%)

TRUE S2 PAIRS


NameError: name 'true_s2_scored' is not defined

In [110]:
# ============================================
# Create scored ground-truth S2 pairs
# ============================================

import pandas as pd
import re
import unicodedata

# Start from the known TRUE S2 pairs
true_s2_scored = gt_s2.copy()

# Normalize text for fuzzy comparison
def normalize_text(x):
    if pd.isna(x):
        return ""
    
    x = str(x).lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    
    return x


# Use the existing detailed information
true_s2_scored = true_s2_scored.merge(
    s1_info,
    on="source1_entity_id",
    how="left"
)

true_s2_scored = true_s2_scored.merge(
    s2_info,
    on="matched_entity_ids",
    how="left"
)

# Check columns
print("Columns:")
print(true_s2_scored.columns.tolist())

print("\nShape:")
print(true_s2_scored.shape)

Columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country']

Shape:
(3693619, 11)


In [112]:
# ============================================
# Score TRUE S2 pairs
# ============================================

from difflib import SequenceMatcher

def similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio() * 100


true_s2_scored["name_score"] = true_s2_scored.apply(
    lambda r: similarity(
        r["s1_name"],
        r["matched_name"]
    ),
    axis=1
)

true_s2_scored["address_score"] = true_s2_scored.apply(
    lambda r: similarity(
        r["s1_address"],
        r["matched_address"]
    ),
    axis=1
)

# Same weighting as your remaining candidates
true_s2_scored["combined_score"] = (
    0.6 * true_s2_scored["name_score"]
    + 0.4 * true_s2_scored["address_score"]
)

print("TRUE S2 PAIRS:", len(true_s2_scored))

display(
    true_s2_scored[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ].describe()
)

TRUE S2 PAIRS: 3693619


,name_score,address_score,combined_score
count,3.693619e+06,3.693619e+06,3.693619e+06
mean,7.799797e+01,7.910375e+01,7.844028e+01
std,2.864231e+01,2.337465e+01,1.993699e+01
min,0.000000e+00,0.000000e+00,1.739130e+00
25%,7.272727e+01,7.118644e+01,7.273657e+01
50%,8.800000e+01,8.709677e+01,8.468702e+01
75%,1.000000e+02,9.473684e+01,9.238896e+01
max,1.000000e+02,1.000000e+02,1.000000e+02


In [113]:
thresholds = [80, 85, 90, 92, 94, 95, 96, 97, 98]

print("=" * 75)
print("THRESHOLD ANALYSIS")
print("=" * 75)

print(
    f"{'Threshold':<12}"
    f"{'True Captured':<18}"
    f"{'True Recall':<18}"
    f"{'Remaining Candidates':<22}"
)

print("-" * 75)

for t in thresholds:

    true_captured = (
        true_s2_scored["combined_score"] >= t
    ).sum()

    true_recall = (
        true_captured /
        len(true_s2_scored) *
        100
    )

    remaining_candidates = (
        remaining["combined_score"] >= t
    ).sum()

    print(
        f"{t:<12}"
        f"{true_captured:<18,}"
        f"{true_recall:<18.2f}%"
        f"{remaining_candidates:<22,}"
    )

THRESHOLD ANALYSIS
Threshold   True Captured     True Recall       Remaining Candidates  
---------------------------------------------------------------------------
80          2,297,560         62.20             %158,897               
85          1,817,321         49.20             %99,629                
90          1,241,447         33.61             %45,486                
92          981,147           26.56             %27,402                
94          711,489           19.26             %13,427                
95          585,022           15.84             %8,194                 
96          465,206           12.59             %3,911                 
97          350,073           9.48              %1,474                 
98          237,186           6.42              %197                   


In [114]:
# ============================================
# STEP 1: Inspect scripts in remaining S2 pairs
# ============================================

import unicodedata
import pandas as pd

def detect_script(text):
    if pd.isna(text):
        return "EMPTY"

    text = str(text)

    scripts = set()

    for ch in text:
        if not ch.isalpha():
            continue

        name = unicodedata.name(ch, "")

        if "LATIN" in name:
            scripts.add("Latin")
        elif "DEVANAGARI" in name:
            scripts.add("Devanagari")
        elif "TELUGU" in name:
            scripts.add("Telugu")
        elif "KANNADA" in name:
            scripts.add("Kannada")
        elif "TAMIL" in name:
            scripts.add("Tamil")
        elif "GUJARATI" in name:
            scripts.add("Gujarati")
        elif "BENGALI" in name:
            scripts.add("Bengali")
        elif "MALAYALAM" in name:
            scripts.add("Malayalam")
        elif "GURMUKHI" in name:
            scripts.add("Gurmukhi")
        else:
            scripts.add("Other")

    if not scripts:
        return "EMPTY"

    return "+".join(sorted(scripts))


remaining["s1_script"] = remaining["s1_name"].apply(detect_script)
remaining["s2_script"] = remaining["matched_name"].apply(detect_script)

print("S1 scripts:")
print(remaining["s1_script"].value_counts())

print("\nS2 scripts:")
print(remaining["s2_script"].value_counts())

S1 scripts:
s1_script
Latin    663020
Name: count, dtype: int64

S2 scripts:
s2_script
Latin               401300
Devanagari          144968
Telugu               21142
Kannada              20227
Tamil                18127
Bengali              16567
Gujarati             16522
Malayalam            10213
Other                 3984
Gurmukhi              3653
Devanagari+Latin      3546
Latin+Telugu           515
Kannada+Latin          479
Latin+Tamil            461
Bengali+Latin          405
Gujarati+Latin         396
Latin+Malayalam        262
Latin+Other             98
Gurmukhi+Latin          92
EMPTY                   63
Name: count, dtype: int64


In [116]:
cross_script = remaining[
    remaining["s1_script"] != remaining["s2_script"]
].copy()

print("Total cross-script candidates:", len(cross_script))

display(
    cross_script[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "name_score",
            "address_score",
            "combined_score",
            "s1_script",
            "s2_script"
        ]
    ].head(30)
)

Total cross-script candidates: 261720


,source1_entity_id,matched_entity_ids,s1_name,matched_name,s1_address,matched_address,name_score,address_score,combined_score,s1_script,s2_script
5,S1-602144510,S2-32696802,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delh...","C-93, DELHI, NEW DELHI, Delhi",8.000000,70.270270,29.794595,Latin,Devanagari
6,S1-602144510,S2-64000798,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delh...","C-93, G/F, RING ROAD, NDSE-II, DELHI, NEW DELH...",8.000000,100.000000,40.200000,Latin,Devanagari
7,S1-559249977,S2-312080751,My Investment Private Limited,माय इन्वेस्टमेंट प्राइवेट लिमिटेड,"C/O Mayank Anilkumar Sing, Chinchani District ...","C/O MAYANK ANILKUMAR SING, PALGHAR, THANE, Mah...",9.836066,80.314961,34.503679,Latin,Devanagari
8,S1-507257254,S2-420623386,Galaxy Consultancy Limited,ಗ್ಯಾಲಕ್ಸಿ ಕನ್ಸಲ್ಟೆನ್ಸಿ ಲಿಮಿಟೆಡ್,"#270 10Th Main 2Nd Block Jayanagar, Bangalore,...","BENGALURU, #G-270 10TH MAIN 2ND BLOCK JAYANAGA...",7.407407,89.830508,36.255493,Latin,Kannada
14,S1-657625699,S2-257177811,My Solutions Private Limited,माय सॉल्यूशंस प्राइवेट लिमिटेड,"Delhi, East Delhi, Delhi, C12 Gf Ganesh Nagar ...","C##12 GF GANESH NAGAR PANDAV NAGAR COMPLEX, DE...",10.344828,62.992126,28.771382,Latin,Devanagari
15,S1-28453582,S2-72442936,Laxmi Foundation Pvt Ltd,லக்ஷ்மி ஃபவுண்டேஷன் பிரைவேட் லிமிடெட்,"No 45 F Karukinil, Amarthaval Koil Street, Kan...","DOOR NO #45 F KARUKINIL, AMARTHAVAL KOIL STREE...",10.526316,96.815287,40.727456,Latin,Tamil
16,S1-480158538,S2-121226226,Ram Logistics Private Limited,राम लॉजिस्टिक्स प्राइवेट लिमिटेड,"B-48, Hanuman Temple, Nr. Park Kiran Garden, U...","B-48, HANUMAN TEMPLE, NR. PARK KIRAN GARDEN, U...",9.836066,100.000000,41.393443,Latin,Devanagari
18,S1-973304639,S2-178208297,Good Finance,गुड फाइनेंस,"D-5, Gno 1, Khn 415, Sanjay Mohlla Bhajanpura,...","D-5, DELHI, NORTH EAST, Delhi",9.090909,46.017699,22.015286,Latin,Devanagari
19,S1-34698027,S2-783389260,Unique Construction Pvt Ltd,यूनिक कंस्ट्रक्शन प्रा. लि.,S.No. 138/1 Green Olive Fl- A/501 Fl-A/501 Nr ...,S.NO. 138/1 GREEN OLIVE FL- A/501 FL-A/501 NR ...,12.000000,100.000000,42.800000,Latin,Devanagari
23,S1-244643323,S2-104793636,Good Life Services Pvt Ltd,गुड लाइफ सर्विसेज प्रा. लि.,"Plot No. 100-101, Sector-35, Hsiidc Udyog Viha...","DOOR NO 100-101/6, GURGAON, SADAR BAZAR, Haryana",16.326531,58.267717,31.005946,Latin,Devanagari


In [117]:
true_s2_scored["s1_script"] = (
    true_s2_scored["s1_name"].apply(detect_script)
)

true_s2_scored["s2_script"] = (
    true_s2_scored["matched_name"].apply(detect_script)
)

true_cross_script = true_s2_scored[
    true_s2_scored["s1_script"] != true_s2_scored["s2_script"]
].copy()

print(
    "True cross-script S2 pairs:",
    len(true_cross_script)
)

display(
    true_cross_script[
        [
            "s1_name",
            "matched_name",
            "name_score",
            "address_score",
            "combined_score",
            "s1_script",
            "s2_script"
        ]
    ].head(30)
)

True cross-script S2 pairs: 344641


,s1_name,matched_name,name_score,address_score,combined_score,s1_script,s2_script
2,Raj Investments LLP,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,0.000000,100.000000,40.000000,Latin,Tamil
6,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,0.000000,81.761006,32.704403,Latin,Devanagari
7,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,0.000000,32.000000,12.800000,Latin,Devanagari
12,Red Ventures Private Limited,रेड वेंचर्स प्राइवेट लिमिटेड,0.000000,38.775510,15.510204,Latin,Devanagari
14,Hotel Enterprises Limited,होटल एंटरप्राइजेज लिमिटेड,0.000000,56.551724,22.620690,Latin,Devanagari
32,Balaji Investment Private Limited,బాలాజీ ఇన్వెస్ట్‌మెంట్ ప్రైవేట్ లిమిటెడ్,0.000000,53.913043,21.565217,Latin,Telugu
35,Swastik Om Solutions LLP,स्वस्तिक ॐ सॉल्यूशंस एलएलपी,0.000000,69.306931,27.722772,Latin,Devanagari
36,Swastik Om Solutions LLP,स्वस्तिक ॐ सॉल्यूशंस एलएलपी,0.000000,93.750000,37.500000,Latin,Devanagari
71,New Solutions,न्यू सॉल्यूशंस,0.000000,79.661017,31.864407,Latin,Devanagari
74,New Solutions,न्यू सॉल्यूशंस,0.000000,79.661017,31.864407,Latin,Devanagari


In [118]:
print("Cross-script related DataFrames:")

for name, obj in globals().items():
    if hasattr(obj, "columns") and isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if (
            "s1_script" in cols
            or "s2_script" in cols
            or "script" in name.lower()
            or "cross" in name.lower()
        ):
            print(f"{name:30} shape={obj.shape}")

Cross-script related DataFrames:
remaining                      shape=(663020, 20)
true_s2_scored                 shape=(3693619, 16)
cross_script                   shape=(261720, 20)
true_cross_script              shape=(344641, 16)


In [119]:
print("=" * 70)
print("CROSS-SCRIPT SCORE ANALYSIS")
print("=" * 70)

print("\nTrue cross-script pairs:", len(true_cross_script))
print("Remaining cross-script candidates:", len(cross_script))

print("\n--- TRUE CROSS-SCRIPT SCORE DISTRIBUTION ---")
display(
    true_cross_script[
        ["name_score", "address_score", "combined_score"]
    ].describe()
)

print("\n--- REMAINING CROSS-SCRIPT SCORE DISTRIBUTION ---")
display(
    cross_script[
        ["name_score", "address_score", "combined_score"]
    ].describe()
)

CROSS-SCRIPT SCORE ANALYSIS

True cross-script pairs: 344641
Remaining cross-script candidates: 261720

--- TRUE CROSS-SCRIPT SCORE DISTRIBUTION ---


,name_score,address_score,combined_score
count,344641.000000,344641.000000,344641.000000
mean,2.626982,74.924692,31.546066
std,13.191214,21.273499,11.845259
min,0.000000,0.000000,1.739130
25%,0.000000,58.974359,23.829787
50%,0.000000,80.000000,32.413793
75%,0.000000,93.333333,37.829457
max,95.652174,100.000000,97.108305



--- REMAINING CROSS-SCRIPT SCORE DISTRIBUTION ---


,name_score,address_score,combined_score
count,261720.000000,261720.000000,261720.000000
mean,11.380514,74.816863,33.583236
std,7.950937,20.151044,8.768197
min,0.000000,0.000000,9.285714
25%,9.090909,59.060403,27.499139
50%,10.000000,78.378378,34.324898
75%,11.320755,92.500000,39.333333
max,95.384615,100.000000,95.422535


In [120]:
thresholds = [50, 60, 65, 70, 75, 80, 85, 90, 92, 94, 95]

print("\n" + "=" * 70)
print("CROSS-SCRIPT THRESHOLD ANALYSIS")
print("=" * 70)

print(
    f"{'Threshold':<12}"
    f"{'True Captured':<18}"
    f"{'True Recall %':<18}"
)

print("-" * 50)

total_true = len(true_cross_script)

for t in thresholds:
    captured = (true_cross_script["combined_score"] >= t).sum()
    recall = captured / total_true * 100

    print(
        f"{t:<12}"
        f"{captured:<18}"
        f"{recall:<18.2f}"
    )


CROSS-SCRIPT THRESHOLD ANALYSIS
Threshold   True Captured     True Recall %     
--------------------------------------------------
50          12894             3.74              
60          10359             3.01              
65          9023              2.62              
70          7521              2.18              
75          5810              1.69              
80          4008              1.16              
85          2227              0.65              
90          770               0.22              
92          379               0.11              
94          129               0.04              
95          58                0.02              


In [121]:
cross_script_low = (
    true_cross_script
    .sort_values("combined_score", ascending=True)
)

display(
    cross_script_low[
        [
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ].head(100)
)

,s1_name,matched_name,s1_address,matched_address,name_score,address_score,combined_score
1130914,Seven Technologies Limited,सेवन टेक्नोलॉजीज लिमिटेड,"108, Bldg-5, Veena Saaz (Veena Swar), Opp. Tha...","GREATER BOMBAY, महाराष्ट्र, #950 108",0.0,4.347826,1.739130
1385516,Modern Global Private Limited,मॉडर्न ग्लोबल प्राइवेट लिमिटेड,"40, Rz Block South Extension Part-1, Uttam Nag...","DIVREPORTINGCIRCLE, 40, दिल्ली",0.0,4.444444,1.777778
3437553,Creative Food Private Limited,ક્રિએટિવ ફૂડ પ્રાઇવેટ લિમિટેડ,"32, First Floor, Krishnacomplex Radhanpur Char...","RAKSA, ગુજરાત, 32",0.0,4.819277,1.927711
2570991,Star Tech Private Limited,स्टार टेक प्राइवेट लिमिटेड,"315, Shivai Plaza, Bhd Ravi Vihar Hotel Lane T...","GREATOR BOMBAY, महाराष्ट्र, 315",0.0,5.084746,2.033898
2969562,Krishna Estate Private Limited,कृष्णा एस्टेट प्राइवेट लिमिटेड,"117, Pocket A-4 Konark Apartments, Kalkaji Ext...","SONIPAT, दिल्ली, 117",0.0,6.741573,2.696629
...,...,...,...,...,...,...,...
1813559,North It Private Limited,নর্থ আইটি প্রাইভেট লিমিটেড,"9A, Chatterjee International Chamber, 9Th Floo...","#9A, HOWRAH, পশ্চিমবঙ্গ",0.0,13.533835,5.413534
2820456,Sree Foundation Private Limited,ಶ್ರೀ ಫೌಂಡೇಶನ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"Lmcpl, 57/1A, 1St Floor, Unit-24, Devarabisana...","Karnataka, LMCPL, BENGALURU",0.0,13.533835,5.413534
1656744,North Technology Pvt Ltd,नॉर्थ टेक्नोलॉजी प्रा. लि.,"Level 10, Hudson And Ganges, Embassy Tech Zone...","Maharashtra, MULSHI, PUNE, LEVEL 10",0.0,13.580247,5.432099
2080386,Surya Foundation Private Limited,सूर्या फाउंडेशन प्राइवेट लिमिटेड,"Nashik, Shriram Niwas, Chhatrapati Colony, Vad...","8, NASHIK, महाराष्ट्र",0.0,13.592233,5.436893


In [125]:
# STEP 4: Cross-script name transliteration
# ------------------------------------------

import re
import pandas as pd

# Install once if needed:
# !pip install indic-transliteration

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


# --------------------------------------------------
# Script ranges
# --------------------------------------------------

SCRIPT_RANGES = {
    "Devanagari": (0x0900, 0x097F),
    "Bengali":    (0x0980, 0x09FF),
    "Gurmukhi":   (0x0A00, 0x0A7F),
    "Gujarati":   (0x0A80, 0x0AFF),
    "Tamil":      (0x0B80, 0x0BFF),
    "Telugu":     (0x0C00, 0x0C7F),
    "Kannada":    (0x0C80, 0x0CFF),
    "Malayalam":  (0x0D00, 0x0D7F),
}


SCRIPT_TO_SANSCRIPT = {
    "Devanagari": sanscript.DEVANAGARI,
    "Bengali":    sanscript.BENGALI,
    "Gurmukhi":   sanscript.GURMUKHI,
    "Gujarati":   sanscript.GUJARATI,
    "Tamil":      sanscript.TAMIL,
    "Telugu":     sanscript.TELUGU,
    "Kannada":    sanscript.KANNADA,
    "Malayalam":  sanscript.MALAYALAM,
}


def get_char_script(ch):
    cp = ord(ch)

    for script, (lo, hi) in SCRIPT_RANGES.items():
        if lo <= cp <= hi:
            return script

    return None


def transliterate_name(text):
    """
    Convert Indian-script portions to Latin while preserving
    existing Latin text, numbers and punctuation.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    output = []
    current_script = None
    current_text = []

    def flush():
        nonlocal current_script, current_text

        if not current_text:
            return

        chunk = "".join(current_text)

        if current_script in SCRIPT_TO_SANSCRIPT:
            try:
                converted = transliterate(
                    chunk,
                    current_script_source,
                    sanscript.ITRANS
                )
            except Exception:
                converted = chunk
        else:
            converted = chunk

        output.append(converted)

        current_text = []

    for ch in text:

        script = get_char_script(ch)

        if script != current_script:

            # Flush previous run
            if current_text:

                if current_script in SCRIPT_TO_SANSCRIPT:
                    current_script_source = SCRIPT_TO_SANSCRIPT[current_script]

                    try:
                        output.append(
                            transliterate(
                                "".join(current_text),
                                current_script_source,
                                sanscript.ITRANS
                            )
                        )
                    except Exception:
                        output.append("".join(current_text))
                else:
                    output.append("".join(current_text))

                current_text = []

            current_script = script

        current_text.append(ch)

    # Flush final run
    if current_text:

        if current_script in SCRIPT_TO_SANSCRIPT:
            current_script_source = SCRIPT_TO_SANSCRIPT[current_script]

            try:
                output.append(
                    transliterate(
                        "".join(current_text),
                        current_script_source,
                        sanscript.ITRANS
                    )
                )
            except Exception:
                output.append("".join(current_text))
        else:
            output.append("".join(current_text))

    return "".join(output)


# --------------------------------------------------
# Test examples
# --------------------------------------------------

test_names = [
    "राज् इन्वेस्टमेंट्स एलएलपी",
    "एसएस फूड प्राइवेट लिमिटेड",
    "బాలాజీ ఇన్వెస్ట్‌మెంట్ ప్రైవేట్ లిమిటెడ్",
    "ಗುರು ಸೊಲ್ಯೂಷನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್",
    "ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி",
    "রেড গ্লোবাল লিমিটেড",
]

for x in test_names:
    print(x)
    print(" -> ", transliterate_name(x))
    print()

राज् इन्वेस्टमेंट्स एलएलपी
 ->  rAj invesTameMTsa elaelapI

एसएस फूड प्राइवेट लिमिटेड
 ->  esaesa phUDa prAiveTa limiTeDa

బాలాజీ ఇన్వెస్ట్‌మెంట్ ప్రైవేట్ లిమిటెడ్
 ->  bAlAjI invèsT‌mèMT praiveT limiTèD

ಗುರು ಸೊಲ್ಯೂಷನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್
 ->  guru sòlyUShans praiveT limiTèD

ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி
 ->  rAj iனvèsDhmèNDhs èlèlbhi

রেড গ্লোবাল লিমিটেড
 ->  reDa glovAla limiTeDa



In [126]:
# Create transliterated S2 names

cross_script = cross_script.copy()

cross_script["matched_name_latin"] = (
    cross_script["matched_name"]
    .fillna("")
    .apply(transliterate_name)
)

display(
    cross_script[
        [
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_script",
            "s2_script"
        ]
    ].head(50)
)

,s1_name,matched_name,matched_name_latin,s1_script,s2_script
5,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,iMTaraneshanala mainejameMTa prAiveTa limiTeDa,Latin,Devanagari
6,International Management Private Limited,इंटरनेशनल मैनेजमेंट प्राइवेट लिमिटेड,iMTaraneshanala mainejameMTa prAiveTa limiTeDa,Latin,Devanagari
7,My Investment Private Limited,माय इन्वेस्टमेंट प्राइवेट लिमिटेड,mAya invesTameMTa prAiveTa limiTeDa,Latin,Devanagari
8,Galaxy Consultancy Limited,ಗ್ಯಾಲಕ್ಸಿ ಕನ್ಸಲ್ಟೆನ್ಸಿ ಲಿಮಿಟೆಡ್,gyAlaksi kansalTènsi limiTèD,Latin,Kannada
14,My Solutions Private Limited,माय सॉल्यूशंस प्राइवेट लिमिटेड,mAya saॉlyUshaMsa prAiveTa limiTeDa,Latin,Devanagari
15,Laxmi Foundation Pvt Ltd,லக்ஷ்மி ஃபவுண்டேஷன் பிரைவேட் லிமிடெட்,lakShmi favuNDheShaன bhiraiveDh limiDhèDh,Latin,Tamil
16,Ram Logistics Private Limited,राम लॉजिस्टिक्स प्राइवेट लिमिटेड,rAma laॉjisTiksa prAiveTa limiTeDa,Latin,Devanagari
18,Good Finance,गुड फाइनेंस,guDa phAineMsa,Latin,Devanagari
19,Unique Construction Pvt Ltd,यूनिक कंस्ट्रक्शन प्रा. लि.,yUnika kaMsTrakshana prA. li.,Latin,Devanagari
23,Good Life Services Pvt Ltd,गुड लाइफ सर्विसेज प्रा. लि.,guDa lAipha sarviseja prA. li.,Latin,Devanagari


In [127]:
from rapidfuzz.fuzz import ratio

def normalize_for_match(x):
    if pd.isna(x):
        return ""

    x = str(x).lower()

    # Replace ITRANS-style punctuation/symbols
    x = x.replace("'", "")
    x = x.replace("~", "")
    x = x.replace(":", "")

    # Keep only alphanumeric characters
    x = re.sub(r"[^a-z0-9]+", " ", x)

    # Collapse spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x


cross_script["s1_name_norm"] = (
    cross_script["s1_name"]
    .apply(normalize_for_match)
)

cross_script["s2_name_latin_norm"] = (
    cross_script["matched_name_latin"]
    .apply(normalize_for_match)
)


cross_script["name_score_translit"] = cross_script.apply(
    lambda r: ratio(
        r["s1_name_norm"],
        r["s2_name_latin_norm"]
    ),
    axis=1
)


display(
    cross_script[
        [
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "name_score",
            "name_score_translit"
        ]
    ].sort_values(
        "name_score_translit",
        ascending=False
    ).head(100)
)

,s1_name,matched_name,matched_name_latin,name_score,name_score_translit
324521,Alpha Agro Private Limited,ಆಲ್ಫಾ ಅಗ್ರೋ Private Limited,AlphA agro Private Limited,66.666667,100.0
511084,Lakshmi Enterprises Private Limited,लक्ष्मी Enterprises Private Limited,lakShmI Enterprises Private Limited,81.159420,100.0
646084,Aditya Consulting LLP,आदित्य Consulting LLP,Aditya Consulting LLP,71.428571,100.0
609499,Shiva Finance Pvt Ltd,శివ Finance Pvt Ltd,shiva Finance Pvt Ltd,80.000000,100.0
335504,Vijay International Limited,വിജയ് International Limited,vijay International Limited,83.018868,100.0
...,...,...,...,...,...
147460,Jain Properties Limited,ಜೈನ್ Properties Limited,jain Properties Limited,84.444444,100.0
479193,Alpha Foundation Private Limited,ಆಲ್ಫಾ Foundation Private Limited,AlphA Foundation Private Limited,85.714286,100.0
299849,Raj Agro,రాజ్ ఆగ్రో,rAj Agro,12.500000,100.0
113668,Shakti Marketing Private Limited,शक्ति Marketing Private Limited,shakti Marketing Private Limited,83.870968,100.0


In [128]:
print("=" * 70)
print("CROSS-SCRIPT NAME TRANSLITERATION IMPROVEMENT")
print("=" * 70)

print("\nOld name score:")
print(cross_script["name_score"].describe())

print("\nNew transliterated name score:")
print(cross_script["name_score_translit"].describe())


print("\nScore >= thresholds")

for t in [50, 60, 70, 75, 80, 85, 90, 95]:

    old_count = (
        cross_script["name_score"] >= t
    ).sum()

    new_count = (
        cross_script["name_score_translit"] >= t
    ).sum()

    print(
        f"{t:>3} | "
        f"Old: {old_count:>8} | "
        f"New: {new_count:>8} | "
        f"Gain: {new_count-old_count:>8}"
    )

CROSS-SCRIPT NAME TRANSLITERATION IMPROVEMENT

Old name score:
count    261720.000000
mean         11.380514
std           7.950937
min           0.000000
25%           9.090909
50%          10.000000
75%          11.320755
max          95.384615
Name: name_score, dtype: float64

New transliterated name score:
count    261720.000000
mean         68.650388
std          13.326899
min           0.000000
25%          63.333333
50%          70.886076
75%          76.712329
max         100.000000
Name: name_score_translit, dtype: float64

Score >= thresholds
 50 | Old:     3711 | New:   246655 | Gain:   242944
 60 | Old:     2547 | New:   215002 | Gain:   212455
 70 | Old:     1838 | New:   141216 | Gain:   139378
 75 | Old:     1533 | New:    87164 | Gain:    85631
 80 | Old:     1141 | New:    38010 | Gain:    36869
 85 | Old:      561 | New:    10141 | Gain:     9580
 90 | Old:      164 | New:     2968 | Gain:     2804
 95 | Old:        1 | New:     1055 | Gain:     1054


In [129]:
cross_script["combined_score_translit"] = (
    0.60 * cross_script["name_score_translit"]
    + 0.40 * cross_script["address_score"]
)

In [130]:
cross_script_best = (
    cross_script
    .sort_values(
        "combined_score_translit",
        ascending=False
    )
)

display(
    cross_script_best[
        [
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].head(100)
)

,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit
167819,Vijay Management Private Limited,ವಿಜಯ್ Management Private Limited,vijay Management Private Limited,"Karnataka, No.119, Bangalore, Sri Puttanna Che...","Karnataka, NO.119, BANGALORE, SRI PUTTANNA CHE...",100.000000,100.000000,100.000000
438429,Global Engineering,ಗ್ಲೋಬಲ್ Engineering,global Engineering,"No 4, 1St Main 1St Block, R T Nagar, Bangalore...","NO 4, 1ST MAIN 1ST BLOCK, R T NAGAR, BANGALORE...",100.000000,100.000000,100.000000
618899,Tirupati Ventures Private Limited,तिरुपति Ventures Private Limited,tirupati Ventures Private Limited,"C/O Rajrani, Near Shiva Temple, Nawabganj, Bar...","C/O RAJRANI, NEAR SHIVA TEMPLE, NAWABGANJ, BAR...",100.000000,100.000000,100.000000
113538,Surya Builders Limited,सूर्या Builders Limited,sUryA Builders Limited,"Shiv Complex, Near Railway Station, Mainpuri, ...","SHIV COMPLEX, NEAR RAILWAY STATION, MAINPURI, ...",100.000000,100.000000,100.000000
108820,Baba Construction LLP,ਬਾਬਾ Construction LLP,bAbA Construction LLP,"179 Guru Nanak Colony, St No-02, Faridkot, Far...","179 GURU NANAK COLONY, ST NO-02, FARIDKOT, FAR...",100.000000,100.000000,100.000000
...,...,...,...,...,...,...,...,...
488679,Hotel Energy Pvt Ltd,হোটেল Energy Pvt Ltd,hoTela Energy Pvt Ltd,"Kamdevpur Delhi Roadsugandha, Hooghly, West Be...","KAMDEVPUR DELHI ROADSUGANDHA, HOOGHLY, West Be...",97.560976,100.000000,98.536585
552846,Red Global Private Limited,रेड Global Private Limited,reDa Global Private Limited,"1205, 12Th Floor, A-Building, Kumar Surabhi, S...","D/1205, 12TH FLOOR, A-BUILDING, KUMAR SURABHI,...",98.113208,99.019608,98.475768
442095,Best Infrastructure,बेस्ट Infrastructure,besTa Infrastructure,"Gangzala. Ward No.13, Saharsa Ps- Saharsa, Sah...","GANGZALA. WARD NO.13, SAHARSA PS- SAHARSA, SAH...",97.435897,100.000000,98.461538
609741,Great Modern Industries Private Limited,ग्रेट Modern Industries Private Limited,greTa Modern Industries Private Limited,"Flat No-692, Dda Flats Pocket-13, Phase-I Dwar...","FLAT NO-692, DDA FLATS POCKET-13, PHASE-I DWAR...",97.435897,100.000000,98.461538


In [135]:
# ============================================================
# STEP 9 — ADD TRANSLITERATION SCORES TO TRUE CROSS-SCRIPT
# ============================================================

print("true_cross_script columns before:")
print(true_cross_script.columns.tolist())

print("\ncross_script columns:")
print(cross_script.columns.tolist())


# Get only the transliteration scores we already calculated
translit_scores = cross_script[
    [
        "source1_entity_id",
        "matched_entity_ids",
        "name_score_translit"
    ]
].copy()


# Remove any accidental duplicate pair rows
translit_scores = translit_scores.drop_duplicates(
    subset=["source1_entity_id", "matched_entity_ids"]
)


# Add transliteration score to true cross-script pairs
true_cross_script = true_cross_script.merge(
    translit_scores,
    on=["source1_entity_id", "matched_entity_ids"],
    how="left"
)


# Recalculate combined score
# Same weighting used earlier:
# 60% name + 40% address

true_cross_script["combined_score_translit"] = (
    0.60 * true_cross_script["name_score_translit"]
    + 0.40 * true_cross_script["address_score"]
)


print("\nAfter merge:")
print(true_cross_script.shape)

print("\nNew columns:")
print(
    true_cross_script[
        [
            "s1_name",
            "matched_name",
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].head(20)
)

true_cross_script columns before:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country', 'name_score', 'address_score', 'combined_score', 's1_script', 's2_script']

cross_script columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country', 'name_s1_norm', 'name_s2_norm', 'address_s1_norm', 'address_s2_norm', 'name_score', 'address_score', 'combined_score', 's1_script', 's2_script', 'matched_name_latin', 's1_name_norm', 's2_name_latin_norm', 'name_score_translit', 'combined_score_translit']

After merge:
(344641, 18)

New columns:
                              s1_name  \
0                 Raj Investments LLP   
1             Ss Food Private Limited   
2             Ss Food Private Limited   
3        Red Ventures Private Limite

In [136]:
print(
    "Missing transliteration scores:",
    true_cross_script["name_score_translit"].isna().sum()
)

print(
    "Missing combined scores:",
    true_cross_script["combined_score_translit"].isna().sum()
)

Missing transliteration scores: 82921
Missing combined scores: 82921


In [138]:
true_cross_script["matched_name_latin"] = (
    true_cross_script["matched_name"]
    .fillna("")
    .apply(transliterate_name)
)

In [139]:
# STEP 11: Recalculate missing cross-script transliteration scores
# ---------------------------------------------------------------

# Work only on rows where transliteration score is missing
missing_mask = cross_script["name_score_translit"].isna()

print("Missing transliteration scores before:", missing_mask.sum())

cross_script.loc[missing_mask, "matched_name_latin"] = (
    cross_script.loc[missing_mask, "matched_name"]
    .fillna("")
    .apply(transliterate_name)
)

# Normalize both names before scoring
cross_script.loc[missing_mask, "s1_name_norm"] = (
    cross_script.loc[missing_mask, "s1_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

cross_script.loc[missing_mask, "s2_name_latin_norm"] = (
    cross_script.loc[missing_mask, "matched_name_latin"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Recalculate name score
cross_script.loc[missing_mask, "name_score_translit"] = (
    cross_script.loc[missing_mask]
    .apply(
        lambda row: ratio(
            row["s1_name_norm"],
            row["s2_name_latin_norm"]
        ),
        axis=1
    )
)

# Recalculate combined score
cross_script.loc[missing_mask, "combined_score_translit"] = (
    0.6 * cross_script.loc[missing_mask, "name_score_translit"]
    + 0.4 * cross_script.loc[missing_mask, "address_score"]
)

print("Missing transliteration scores after:",
      cross_script["name_score_translit"].isna().sum())

print("Missing combined scores after:",
      cross_script["combined_score_translit"].isna().sum())

Missing transliteration scores before: 0
Missing transliteration scores after: 0
Missing combined scores after: 0


In [140]:
print("Cross-script shape:", cross_script.shape)

print("\nScore statistics:")
print(
    cross_script[
        [
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].describe()
)

print("\nMissing values:")
print(
    cross_script[
        [
            "matched_name_latin",
            "name_score_translit",
            "combined_score_translit"
        ]
    ].isna().sum()
)

Cross-script shape: (261720, 25)

Score statistics:
       name_score_translit  address_score  combined_score_translit
count        261720.000000  261720.000000            261720.000000
mean             68.650388      74.816863                71.116978
std              13.326899      20.151044                11.339909
min               0.000000       0.000000                 8.571429
25%              63.333333      59.060403                64.310345
50%              70.886076      78.378378                72.313842
75%              76.712329      92.500000                79.473684
max             100.000000     100.000000               100.000000

Missing values:
matched_name_latin         0
name_score_translit        0
combined_score_translit    0
dtype: int64


In [141]:
print("Missing transliteration scores after:",
      cross_script["name_score_translit"].isna().sum())

print("Missing combined scores after:",
      cross_script["combined_score_translit"].isna().sum())

Missing transliteration scores after: 0
Missing combined scores after: 0


In [142]:
# STEP 12: Update true cross-script pairs with transliteration scores
# ------------------------------------------------------------------

score_cols = [
    "source1_entity_id",
    "matched_entity_ids",
    "name_score_translit",
    "combined_score_translit"
]

true_cross_script = true_cross_script.drop(
    columns=["name_score_translit", "combined_score_translit"],
    errors="ignore"
)

true_cross_script = true_cross_script.merge(
    cross_script[score_cols],
    on=["source1_entity_id", "matched_entity_ids"],
    how="left"
)

print("Shape:", true_cross_script.shape)

print("\nColumns:")
print(true_cross_script.columns.tolist())

print("\nMissing scores:")
print(
    true_cross_script[
        ["name_score_translit", "combined_score_translit"]
    ].isna().sum()
)

Shape: (344641, 19)

Columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country', 'name_score', 'address_score', 'combined_score', 's1_script', 's2_script', 'matched_name_latin', 'name_score_translit', 'combined_score_translit']

Missing scores:
name_score_translit        82921
combined_score_translit    82921
dtype: int64


In [143]:
print("=" * 70)
print("TRUE CROSS-SCRIPT — TRANSLITERATED SCORE DISTRIBUTION")
print("=" * 70)

print(
    true_cross_script[
        [
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].describe()
)

TRUE CROSS-SCRIPT — TRANSLITERATED SCORE DISTRIBUTION
       name_score_translit  address_score  combined_score_translit
count        261720.000000  344641.000000            261720.000000
mean             68.650388      74.924692                71.116978
std              13.326899      21.273499                11.339909
min               0.000000       0.000000                 8.571429
25%              63.333333      58.974359                64.310345
50%              70.886076      80.000000                72.313842
75%              76.712329      93.333333                79.473684
max             100.000000     100.000000               100.000000


In [144]:
# STEP 14: Threshold analysis after transliteration
# --------------------------------------------------

total_true = len(true_cross_script)

thresholds = [50, 60, 70, 75, 80, 85, 90, 92, 94, 95, 96, 97, 98]

print("=" * 80)
print("CROSS-SCRIPT TRANSLITERATION THRESHOLD ANALYSIS")
print("=" * 80)

print(
    f"{'Threshold':<12}"
    f"{'True Captured':<18}"
    f"{'True Recall %':<18}"
)

print("-" * 50)

for t in thresholds:

    captured = (
        true_cross_script["combined_score_translit"] >= t
    ).sum()

    recall = captured / total_true * 100

    print(
        f"{t:<12}"
        f"{captured:<18}"
        f"{recall:<18.2f}"
    )

CROSS-SCRIPT TRANSLITERATION THRESHOLD ANALYSIS
Threshold   True Captured     True Recall %     
--------------------------------------------------
50          252013            73.12             
60          222624            64.60             
70          151378            43.92             
75          106570            30.92             
80          61572             17.87             
85          22059             6.40              
90          3542              1.03              
92          1509              0.44              
94          714               0.21              
95          478               0.14              
96          334               0.10              
97          214               0.06              
98          133               0.04              


In [145]:
# STEP 15: Score ALL true cross-script pairs
# -------------------------------------------

true_cross_script["matched_name_latin"] = (
    true_cross_script["matched_name"]
    .fillna("")
    .apply(transliterate_name)
)

true_cross_script["s1_name_norm"] = (
    true_cross_script["s1_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

true_cross_script["matched_name_latin_norm"] = (
    true_cross_script["matched_name_latin"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

true_cross_script["name_score_translit"] = (
    true_cross_script.apply(
        lambda row: ratio(
            row["s1_name_norm"],
            row["matched_name_latin_norm"]
        ),
        axis=1
    )
)

true_cross_script["combined_score_translit"] = (
    0.6 * true_cross_script["name_score_translit"]
    + 0.4 * true_cross_script["address_score"]
)

print("Shape:", true_cross_script.shape)

print("\nMissing scores:")
print(
    true_cross_script[
        ["name_score_translit", "combined_score_translit"]
    ].isna().sum()
)

Shape: (344641, 21)

Missing scores:
name_score_translit        0
combined_score_translit    0
dtype: int64


In [146]:
print(
    true_cross_script[
        [
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].describe()
)

       name_score_translit  address_score  combined_score_translit
count        344641.000000  344641.000000            344641.000000
mean             68.692936      74.924692                71.185638
std              12.938073      21.273499                11.567723
min               0.000000       0.000000                12.857143
25%              62.857143      58.974359                63.950311
50%              70.833333      80.000000                72.521515
75%              76.923077      93.333333                79.984556
max             100.000000     100.000000               100.000000


In [147]:
thresholds = [50, 60, 70, 75, 80, 85, 90, 92, 94, 95, 96, 97, 98]

total_true = len(true_cross_script)

print("=" * 80)
print("CROSS-SCRIPT TRANSLITERATION THRESHOLD ANALYSIS")
print("=" * 80)

print(
    f"{'Threshold':<12}"
    f"{'True Captured':<18}"
    f"{'True Recall %':<18}"
)

print("-" * 50)

for t in thresholds:

    captured = (
        true_cross_script["combined_score_translit"] >= t
    ).sum()

    recall = captured / total_true * 100

    print(
        f"{t:<12}"
        f"{captured:<18}"
        f"{recall:<18.2f}"
    )

CROSS-SCRIPT TRANSLITERATION THRESHOLD ANALYSIS
Threshold   True Captured     True Recall %     
--------------------------------------------------
50          329238            95.53             
60          288486            83.71             
70          200185            58.09             
75          144200            41.84             
80          86105             24.98             
85          32572             9.45              
90          6249              1.81              
92          2938              0.85              
94          1377              0.40              
95          874               0.25              
96          577               0.17              
97          365               0.11              
98          207               0.06              


In [148]:
# STEP 18: Transliterate and score all cross-script candidates
# ------------------------------------------------------------

cross_script["matched_name_latin"] = (
    cross_script["matched_name"]
    .fillna("")
    .apply(transliterate_name)
)

cross_script["s1_name_norm"] = (
    cross_script["s1_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

cross_script["matched_name_latin_norm"] = (
    cross_script["matched_name_latin"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

cross_script["name_score_translit"] = (
    cross_script.apply(
        lambda row: ratio(
            row["s1_name_norm"],
            row["matched_name_latin_norm"]
        ),
        axis=1
    )
)

cross_script["combined_score_translit"] = (
    0.6 * cross_script["name_score_translit"]
    + 0.4 * cross_script["address_score"]
)

print("Shape:", cross_script.shape)

print("\nMissing scores:")
print(
    cross_script[
        ["name_score_translit", "combined_score_translit"]
    ].isna().sum()
)

Shape: (261720, 26)

Missing scores:
name_score_translit        0
combined_score_translit    0
dtype: int64


In [149]:
print(
    cross_script[
        [
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].describe()
)

       name_score_translit  address_score  combined_score_translit
count        261720.000000  261720.000000            261720.000000
mean             68.437306      74.816863                70.989129
std              12.823218      20.151044                11.128637
min               0.000000       0.000000                14.948898
25%              62.686567      59.060403                64.048318
50%              70.588235      78.378378                72.067355
75%              76.666667      92.500000                79.344262
max             100.000000     100.000000               100.000000


In [150]:
# STEP 19: Compare true vs candidate cross-script scores
# ------------------------------------------------------

thresholds = [50, 60, 70, 75, 80, 85, 90, 92, 94, 95, 96, 97, 98]

total_true = len(true_cross_script)
total_candidates = len(cross_script)

print("=" * 100)
print("CROSS-SCRIPT TRUE vs CANDIDATE THRESHOLD ANALYSIS")
print("=" * 100)

print(
    f"{'Threshold':<12}"
    f"{'True Captured':<18}"
    f"{'True Recall %':<18}"
    f"{'Candidates Passing':<22}"
    f"{'Candidate %':<15}"
)

print("-" * 100)

for t in thresholds:

    true_captured = (
        true_cross_script["combined_score_translit"] >= t
    ).sum()

    candidates_passing = (
        cross_script["combined_score_translit"] >= t
    ).sum()

    true_recall = true_captured / total_true * 100
    candidate_pct = candidates_passing / total_candidates * 100

    print(
        f"{t:<12}"
        f"{true_captured:<18}"
        f"{true_recall:<18.2f}"
        f"{candidates_passing:<22}"
        f"{candidate_pct:<15.2f}"
    )

CROSS-SCRIPT TRUE vs CANDIDATE THRESHOLD ANALYSIS
Threshold   True Captured     True Recall %     Candidates Passing    Candidate %    
----------------------------------------------------------------------------------------------------
50          329238            95.53             251421                96.06          
60          288486            83.71             221172                84.51          
70          200185            58.09             149183                57.00          
75          144200            41.84             104591                39.96          
80          86105             24.98             60397                 23.08          
85          32572             9.45              21675                 8.28           
90          6249              1.81              3505                  1.34           
92          2938              0.85              1495                  0.57           
94          1377              0.40              702                   0.27 

In [151]:
# STEP 20: Inspect strongest cross-script candidates
# --------------------------------------------------

cols = [
    "source1_entity_id",
    "matched_entity_ids",
    "s1_name",
    "matched_name",
    "matched_name_latin",
    "s1_address",
    "matched_address",
    "name_score_translit",
    "address_score",
    "combined_score_translit"
]

display(
    cross_script
    .sort_values("combined_score_translit", ascending=False)
    [cols]
    .head(100)
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit
291405,S1-147882651,S2-67009607,Jain Energy Private Limited,జైన్ Energy Private Limited,jain Energy Private Limited,"6-21/1/1, Kukatpally Y Junction Moosapet, Hyde...","6-21/1/1, KUKATPALLY Y JUNCTION MOOSAPET, HYDE...",100.000000,100.000000,100.000000
219215,S1-670295316,S2-869031009,Surya Guru Trading Private Limited,సూర్య Guru Trading Private Limited,sUrya Guru Trading Private Limited,"Flat No 502, Kaizenresidency, Hig 13 & 14 Kphb...","FLAT NO 502, KAIZENRESIDENCY, HIG 13 & 14 KPHB...",100.000000,100.000000,100.000000
625263,S1-436440403,S2-995045138,Shiva Trading,शिवा Trading,shivA Trading,"Du- 176 Pitampura, Delhi, North West, Delhi","DU- 176 PITAMPURA, DELHI, NORTH WEST, Delhi",100.000000,100.000000,100.000000
57743,S1-136489621,S2-343737045,Shiva Foundation Private Limited,शिवा Foundation Private Limited,shivA Foundation Private Limited,"Sr No 32/13/1, Row House 101, Sai Shraddha Bun...","SR NO 32/13/1, ROW HOUSE 101, SAI SHRADDHA BUN...",100.000000,100.000000,100.000000
56643,S1-551643922,S2-10097846,Alpha Agro,આલ્ફા Agro,AlphA Agro,"Office No 121, 1St Floor, Sahajanand Luxuria, ...","OFFICE NO 121, 1ST FLOOR, SAHAJANAND LUXURIA, ...",100.000000,100.000000,100.000000
...,...,...,...,...,...,...,...,...,...,...
609741,S1-789731765,S2-506218604,Great Modern Industries Private Limited,ग्रेट Modern Industries Private Limited,greTa Modern Industries Private Limited,"Flat No-692, Dda Flats Pocket-13, Phase-I Dwar...","FLAT NO-692, DDA FLATS POCKET-13, PHASE-I DWAR...",97.435897,100.000000,98.461538
442095,S1-717148219,S2-189611704,Best Infrastructure,बेस्ट Infrastructure,besTa Infrastructure,"Gangzala. Ward No.13, Saharsa Ps- Saharsa, Sah...","GANGZALA. WARD NO.13, SAHARSA PS- SAHARSA, SAH...",97.435897,100.000000,98.461538
612105,S1-611180125,S2-105583678,Gujarat Investments,गुजरात Investments,gujarAta Investments,"C06/01/1:3 Sector No 04 Cbd Belapur Washi, Nav...","C06/01/1:3 SECTOR NO 04 CBD BELAPUR WASHI, NAV...",97.435897,100.000000,98.461538
539077,S1-202953391,S2-654210712,Gujarat Consultancy,गुजरात Consultancy,gujarAta Consultancy,"12/132 Geeta Colony Gandhi Nagar, Delhi, East ...","#12/132 GEETA COLONY GANDHI NAGAR, DELHI, EAST...",97.435897,100.000000,98.461538


In [152]:
display(
    cross_script[
        (cross_script["combined_score_translit"] >= 70) &
        (cross_script["combined_score_translit"] < 80)
    ]
    .sort_values("combined_score_translit", ascending=False)
    [cols]
    .head(100)
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit
555797,S1-855716763,S2-236482785,Supreme Projects Private Limited,सुप्रीम प्रोजेक्ट्स प्राइवेट लिमिटेड,suprIma projekTsa prAiveTa limiTeDa,"Pali Agolpurkadipur Dostpur Road Sultanpur, Ut...",DOOR NO 3-710 PALI AGOLPURKADIPUR DOSTPUR ROAD...,80.597015,79.096045,79.996627
178330,S1-205226738,S2-374065691,Surya Logistics Private Limited,সূর্য লজিস্টিকস প্রাইভেট লিমিটেড,sUrya lajisTikasa prAibheTa limiTeDa,"Elegant Towers 224A, Acharya Jagdish Chandra B...","ELEGANT TOWERS 224A, ACHARYA JAGDISH CHANDRA B...",74.626866,88.050314,79.996245
338222,S1-277063950,S2-90469192,Shyam Developers LLP,શ્યામ ડેવલપર્સ એલએલપી,shyAma Devalaparsa elaelapI,"Plot No. 96, 97 And 98, Rs. No. 40/1 P 1, Gold...","DOOR NO 96, 97 AND 98, RS. NO. 40/1 P 1, GOLDE...",72.340426,91.479821,79.996184
506912,S1-244499461,S2-680484552,Universal Estate Private Limited,ಯುನಿವರ್ಸಲ್ ಎಸ್ಟೇಟ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,yunivarsal èsTeT praiveT limiTèD,"202, 2Nd Floor, Naya Gruha Apts Rachenahalli, ...","Karnataka, 2ND FLOOR, NAYA GRUHA APTS RACHENAH...",78.125000,82.802548,79.996019
31094,S1-517331417,S2-242193420,Al Impex Private Limited,अल इम्पेक्स प्राइवेट लिमिटेड,ala impeksa prAiveTa limiTeDa,"2Nd Floor Above Max Mart Akharaghat Road, Muza...",DOOR NO 2ND FLOOR ABOVE MAX MART AKHARAGHAT RO...,79.245283,81.118881,79.994722
...,...,...,...,...,...,...,...,...,...,...
61925,S1-960564491,S2-512855322,Smart Care Private Limited,स्मार्ट केयर प्राइवेट लिमिटेड,smArTa keyara prAiveTa limiTeDa,"602 Wing A Jangid Heights Chsl, Nr Vidyapeeth ...","H.NO 602 WING A JANGID HEIGHTS CHSL, NR VIDYAP...",77.192982,84.146341,79.974326
95755,S1-545169300,S2-121502087,Premier Management Private Limited,प्रीमियर मैनेजमेंट प्राइवेट लिमिटेड,prImiyara mainejameMTa prAiveTa limiTeDa,"165-D Aravali Apartments, Sector -56, Noida, G...","165-D ARAVALI APARTMENTS, GAUTAM BUDH NAGAR, G...",72.972973,90.476190,79.974260
200833,S1-50383319,S2-620174825,Tirupati Logistics Private Limited,ਤਿਰੂਪਤੀ ਲੌਜਿਸਟਿਕਸ ਪ੍ਰਾਈਵੇਟ ਲਿਮਟਿਡ,tirUpatI laujisaTikasa prAIveTa limaTiDa,"Muktsar, Punjab, 1281 K. Thande Wala Road, Muk...","MUKTSAR, #1281 K. THANDE WALA ROAD, Punjab, B ...",72.972973,90.476190,79.974260
543431,S1-857548759,S2-696412183,Surya Media Private Limited,सूर्य मीडिया प्राइवेट लिमिटेड,sUrya mIDiyA prAiveTa limiTeDa,"Delhi, East Delhi, 3Rd Floor, Plot No.76 & 77,...","OFFICE NO.301, 3RD FLOOR, PLOT NO.76 & 77, KH ...",84.210526,73.619632,79.974169


In [153]:
# ============================================================
# STEP 21: HIGH-CONFIDENCE CROSS-SCRIPT MATCH INSPECTION
# ============================================================

THRESHOLD = 80

high_conf_cross = cross_script[
    cross_script["combined_score_translit"] >= THRESHOLD
].copy()

print("=" * 100)
print(f"CROSS-SCRIPT HIGH-CONFIDENCE MATCHES — SCORE >= {THRESHOLD}")
print("=" * 100)

print("Shape:", high_conf_cross.shape)

print("\nScore statistics:")
print(
    high_conf_cross[
        [
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ].describe()
)

# Display the most convincing matches
display(
    high_conf_cross[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ]
    .sort_values("combined_score_translit", ascending=False)
    .head(100)
)

CROSS-SCRIPT HIGH-CONFIDENCE MATCHES — SCORE >= 80
Shape: (60397, 26)

Score statistics:
       name_score_translit  address_score  combined_score_translit
count         60397.000000   60397.000000             60397.000000
mean             77.766204      94.004471                84.261510
std               6.079962       7.119952                 3.256511
min              66.666667      50.000000                80.000000
25%              73.469388      90.058480                81.683453
50%              77.419355      96.428571                83.636364
75%              81.355932     100.000000                86.153846
max             100.000000     100.000000               100.000000


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit
349257,S1-317518698,S2-603572544,Tirupati Marketing Private Limited,तिरुपति Marketing Private Limited,tirupati Marketing Private Limited,"3-4, 'Aishwarya Sankul', S. No. 17, G. A. Kulk...","3-4, 'AISHWARYA SANKUL', S. NO. 17, G. A. KULK...",100.000000,100.000000,100.000000
304685,S1-124302163,S2-379692379,Shakti Builders Private Limited,शक्ति Builders Private Limited,shakti Builders Private Limited,"C-5 Maharani Bagh, New Delhi, South Delhi, Delhi","C-5 MAHARANI BAGH, NEW DELHI, SOUTH DELHI, Delhi",100.000000,100.000000,100.000000
401029,S1-160667531,S2-715261542,Surya Sree Solutions Limited,सूर्या Sree Solutions Limited,sUryA Sree Solutions Limited,F No 403 M/S Aryadurga Bankar Enterprises Mulu...,F NO 403 M/S ARYADURGA BANKAR ENTERPRISES MULU...,100.000000,100.000000,100.000000
47179,S1-999303555,S2-67942461,Guru Foundation Limited,ಗುರು Foundation Limited,guru Foundation Limited,"No.24, Saberi Complex, Residency Road, Bangalo...","NO.24, SABERI COMPLEX, RESIDENCY ROAD, BANGALO...",100.000000,100.000000,100.000000
326213,S1-327871566,S2-727527025,Surya Infotech Pvt Ltd,সূর্য Infotech Pvt Ltd,sUrya Infotech Pvt Ltd,"5, Puran Chand Nahar Avenue, Kolkata, Kolkata,...","5, PURAN CHAND NAHAR AVENUE, KOLKATA, KOLKATA,...",100.000000,100.000000,100.000000
...,...,...,...,...,...,...,...,...,...,...
442095,S1-717148219,S2-189611704,Best Infrastructure,बेस्ट Infrastructure,besTa Infrastructure,"Gangzala. Ward No.13, Saharsa Ps- Saharsa, Sah...","GANGZALA. WARD NO.13, SAHARSA PS- SAHARSA, SAH...",97.435897,100.000000,98.461538
612105,S1-611180125,S2-105583678,Gujarat Investments,गुजरात Investments,gujarAta Investments,"C06/01/1:3 Sector No 04 Cbd Belapur Washi, Nav...","C06/01/1:3 SECTOR NO 04 CBD BELAPUR WASHI, NAV...",97.435897,100.000000,98.461538
539077,S1-202953391,S2-654210712,Gujarat Consultancy,गुजरात Consultancy,gujarAta Consultancy,"12/132 Geeta Colony Gandhi Nagar, Delhi, East ...","#12/132 GEETA COLONY GANDHI NAGAR, DELHI, EAST...",97.435897,100.000000,98.461538
609741,S1-789731765,S2-506218604,Great Modern Industries Private Limited,ग्रेट Modern Industries Private Limited,greTa Modern Industries Private Limited,"Flat No-692, Dda Flats Pocket-13, Phase-I Dwar...","FLAT NO-692, DDA FLATS POCKET-13, PHASE-I DWAR...",97.435897,100.000000,98.461538


In [154]:
# ============================================================
# STEP 22: BORDERLINE MATCH INSPECTION
# ============================================================

borderline_cross = cross_script[
    (cross_script["combined_score_translit"] >= 70) &
    (cross_script["combined_score_translit"] < 80)
].copy()

print("=" * 100)
print("CROSS-SCRIPT BORDERLINE MATCHES — 70 <= SCORE < 80")
print("=" * 100)

print("Shape:", borderline_cross.shape)

display(
    borderline_cross[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ]
    .sort_values("combined_score_translit", ascending=False)
    .head(100)
)

CROSS-SCRIPT BORDERLINE MATCHES — 70 <= SCORE < 80
Shape: (88786, 26)


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit
555797,S1-855716763,S2-236482785,Supreme Projects Private Limited,सुप्रीम प्रोजेक्ट्स प्राइवेट लिमिटेड,suprIma projekTsa prAiveTa limiTeDa,"Pali Agolpurkadipur Dostpur Road Sultanpur, Ut...",DOOR NO 3-710 PALI AGOLPURKADIPUR DOSTPUR ROAD...,80.597015,79.096045,79.996627
178330,S1-205226738,S2-374065691,Surya Logistics Private Limited,সূর্য লজিস্টিকস প্রাইভেট লিমিটেড,sUrya lajisTikasa prAibheTa limiTeDa,"Elegant Towers 224A, Acharya Jagdish Chandra B...","ELEGANT TOWERS 224A, ACHARYA JAGDISH CHANDRA B...",74.626866,88.050314,79.996245
338222,S1-277063950,S2-90469192,Shyam Developers LLP,શ્યામ ડેવલપર્સ એલએલપી,shyAma Devalaparsa elaelapI,"Plot No. 96, 97 And 98, Rs. No. 40/1 P 1, Gold...","DOOR NO 96, 97 AND 98, RS. NO. 40/1 P 1, GOLDE...",72.340426,91.479821,79.996184
506912,S1-244499461,S2-680484552,Universal Estate Private Limited,ಯುನಿವರ್ಸಲ್ ಎಸ್ಟೇಟ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,yunivarsal èsTeT praiveT limiTèD,"202, 2Nd Floor, Naya Gruha Apts Rachenahalli, ...","Karnataka, 2ND FLOOR, NAYA GRUHA APTS RACHENAH...",78.125000,82.802548,79.996019
31094,S1-517331417,S2-242193420,Al Impex Private Limited,अल इम्पेक्स प्राइवेट लिमिटेड,ala impeksa prAiveTa limiTeDa,"2Nd Floor Above Max Mart Akharaghat Road, Muza...",DOOR NO 2ND FLOOR ABOVE MAX MART AKHARAGHAT RO...,79.245283,81.118881,79.994722
...,...,...,...,...,...,...,...,...,...,...
61925,S1-960564491,S2-512855322,Smart Care Private Limited,स्मार्ट केयर प्राइवेट लिमिटेड,smArTa keyara prAiveTa limiTeDa,"602 Wing A Jangid Heights Chsl, Nr Vidyapeeth ...","H.NO 602 WING A JANGID HEIGHTS CHSL, NR VIDYAP...",77.192982,84.146341,79.974326
95755,S1-545169300,S2-121502087,Premier Management Private Limited,प्रीमियर मैनेजमेंट प्राइवेट लिमिटेड,prImiyara mainejameMTa prAiveTa limiTeDa,"165-D Aravali Apartments, Sector -56, Noida, G...","165-D ARAVALI APARTMENTS, GAUTAM BUDH NAGAR, G...",72.972973,90.476190,79.974260
200833,S1-50383319,S2-620174825,Tirupati Logistics Private Limited,ਤਿਰੂਪਤੀ ਲੌਜਿਸਟਿਕਸ ਪ੍ਰਾਈਵੇਟ ਲਿਮਟਿਡ,tirUpatI laujisaTikasa prAIveTa limaTiDa,"Muktsar, Punjab, 1281 K. Thande Wala Road, Muk...","MUKTSAR, #1281 K. THANDE WALA ROAD, Punjab, B ...",72.972973,90.476190,79.974260
543431,S1-857548759,S2-696412183,Surya Media Private Limited,सूर्य मीडिया प्राइवेट लिमिटेड,sUrya mIDiyA prAiveTa limiTeDa,"Delhi, East Delhi, 3Rd Floor, Plot No.76 & 77,...","OFFICE NO.301, 3RD FLOOR, PLOT NO.76 & 77, KH ...",84.210526,73.619632,79.974169


In [155]:
# ============================================================
# STEP 23: CROSS-SCRIPT SCORE BAND ANALYSIS
# ============================================================

import pandas as pd

score_col = "combined_score_translit"

bins = [0, 50, 60, 70, 75, 80, 85, 90, 95, 100.000001]
labels = [
    "<50",
    "50-60",
    "60-70",
    "70-75",
    "75-80",
    "80-85",
    "85-90",
    "90-95",
    "95-100"
]

# ------------------------------------------------------------
# Candidate distribution
# ------------------------------------------------------------

candidate_bands = pd.cut(
    cross_script[score_col],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

candidate_counts = candidate_bands.value_counts(
    sort=False
)

# ------------------------------------------------------------
# True-pair distribution
# ------------------------------------------------------------

true_bands = pd.cut(
    true_cross_script[score_col],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

true_counts = true_bands.value_counts(
    sort=False
)

# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

band_analysis = pd.DataFrame({
    "Score Band": labels,
    "Candidates": candidate_counts.values,
    "True Pairs": true_counts.values
})

band_analysis["Candidate %"] = (
    band_analysis["Candidates"] /
    len(cross_script) * 100
)

band_analysis["True %"] = (
    band_analysis["True Pairs"] /
    len(true_cross_script) * 100
)

print("=" * 100)
print("CROSS-SCRIPT SCORE BAND ANALYSIS")
print("=" * 100)

display(band_analysis)

CROSS-SCRIPT SCORE BAND ANALYSIS


,Score Band,Candidates,True Pairs,Candidate %,True %
0,<50,10299,15403,3.935122,4.469288
1,50-60,30249,40752,11.557772,11.824478
2,60-70,71989,88301,27.506113,25.621154
3,70-75,44592,55985,17.038056,16.244440
4,75-80,44194,58095,16.885985,16.856671
5,80-85,38722,53533,14.795201,15.532975
6,85-90,18170,26323,6.942534,7.637803
7,90-95,3036,5375,1.160018,1.559594
8,95-100,469,874,0.179199,0.253597


In [156]:
# ============================================================
# STEP 24: CUMULATIVE CROSS-SCRIPT THRESHOLD METRICS
# ============================================================

thresholds = [50, 60, 70, 75, 80, 85, 90, 95]

total_candidates = len(cross_script)
total_true = len(true_cross_script)

results = []

for t in thresholds:

    candidate_mask = (
        cross_script[score_col] >= t
    )

    true_mask = (
        true_cross_script[score_col] >= t
    )

    candidates = candidate_mask.sum()
    true_captured = true_mask.sum()

    recall = (
        true_captured / total_true * 100
        if total_true > 0 else 0
    )

    candidate_rate = (
        candidates / total_candidates * 100
        if total_candidates > 0 else 0
    )

    results.append({
        "Threshold": t,
        "Candidates": candidates,
        "True Captured": true_captured,
        "True Recall %": recall,
        "Candidate %": candidate_rate
    })

threshold_analysis = pd.DataFrame(results)

print("=" * 100)
print("CROSS-SCRIPT CUMULATIVE THRESHOLD METRICS")
print("=" * 100)

display(threshold_analysis)

CROSS-SCRIPT CUMULATIVE THRESHOLD METRICS


,Threshold,Candidates,True Captured,True Recall %,Candidate %
0,50,251421,329238,95.530712,96.064878
1,60,221172,288486,83.706233,84.507107
2,70,149183,200185,58.085080,57.000993
3,75,104591,144200,41.840640,39.962937
4,80,60397,86105,24.983969,23.076952
5,85,21675,32572,9.450994,8.281751
6,90,3505,6249,1.813191,1.339217
7,95,469,874,0.253597,0.179199


In [157]:
# ============================================================
# STEP 25: MULTIPLE-CANDIDATE ANALYSIS
# ============================================================

THRESHOLD = 80

qualified = cross_script[
    cross_script["combined_score_translit"] >= THRESHOLD
].copy()

print("=" * 100)
print(f"MULTIPLE-CANDIDATE ANALYSIS — SCORE >= {THRESHOLD}")
print("=" * 100)

# Number of candidates per S1 entity
candidate_counts = (
    qualified
    .groupby("source1_entity_id")
    .size()
    .reset_index(name="candidate_count")
)

print("\nS1 entities with qualified candidates:", len(candidate_counts))

print("\nCandidate-count distribution:")
print(
    candidate_counts["candidate_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nMaximum candidates for one S1 entity:",
      candidate_counts["candidate_count"].max())

# Entities having more than one candidate
multiple_candidates = candidate_counts[
    candidate_counts["candidate_count"] > 1
].copy()

print("\nS1 entities with >1 candidate:",
      len(multiple_candidates))

print("\nTotal qualified candidate rows:",
      len(qualified))

print("\nTop S1 entities with many candidates:")
display(
    multiple_candidates
    .sort_values("candidate_count", ascending=False)
    .head(20)
)

MULTIPLE-CANDIDATE ANALYSIS — SCORE >= 80

S1 entities with qualified candidates: 50811

Candidate-count distribution:
candidate_count
1    42206
2     7691
3      850
4       61
5        3
Name: count, dtype: int64

Maximum candidates for one S1 entity: 5

S1 entities with >1 candidate: 8605

Total qualified candidate rows: 60397

Top S1 entities with many candidates:


,source1_entity_id,candidate_count
8163,S1-246004420,5
25013,S1-540969303,5
40863,S1-82443192,5
6202,S1-210692425,4
50731,S1-998568047,4
23032,S1-506220317,4
16827,S1-39811475,4
7151,S1-227874238,4
9392,S1-267533518,4
13307,S1-336541763,4


In [158]:
# ============================================================
# STEP 26: INSPECT MULTIPLE MATCHES
# ============================================================

top_ids = multiple_candidates.nlargest(
    10,
    "candidate_count"
)["source1_entity_id"]

multiple_examples = qualified[
    qualified["source1_entity_id"].isin(top_ids)
].copy()

display(
    multiple_examples[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit"
        ]
    ]
    .sort_values(
        ["source1_entity_id", "combined_score_translit"],
        ascending=[True, False]
    )
)

,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit
451382,S1-116730104,S2-949242721,Gold Baba Builders Private Limited,गोल्ड बाबा बिल्डर्स प्राइवेट लिमिटेड,golDa bAbA bilDarsa prAiveTa limiTeDa,"C-804, South End Homes Opp. Gyan Vihar Univers...","C-804, SOUTH END HOMES OPP. GYAN VIHAR UNIVERS...",84.507042,93.670886,88.172580
451383,S1-116730104,S2-525738619,Gold Baba Builders Private Limited,गोल्ड बाबा बिल्डर्स प्राइवेट लिमिटेड,golDa bAbA bilDarsa prAiveTa limiTeDa,"C-804, South End Homes Opp. Gyan Vihar Univers...","C-804, SOUTH END HOMES OPP. GYAN VIHAR UNIVERS...",84.507042,86.624204,85.353907
451384,S1-116730104,S2-581850772,Gold Baba Builders Private Limited,गोल्ड बाबा बिल्डर्स प्राइवेट लिमिटेड,golDa bAbA bilDarsa prAiveTa limiTeDa,"C-804, South End Homes Opp. Gyan Vihar Univers...","Rajasthan, C-804, JAIPUR, SOUTH END HOMES OPP....",84.507042,81.012658,83.109289
451381,S1-116730104,S2-373279353,Gold Baba Builders Private Limited,गोल्ड बाबा बिल्डर्स प्राइवेट लिमिटेड,golDa bAbA bilDarsa prAiveTa limiTeDa,"C-804, South End Homes Opp. Gyan Vihar Univers...","Rajasthan, SOUTH END HOMES OPP. GYAN VIHAR UNI...",84.507042,79.746835,82.602960
341290,S1-136248523,S2-582148603,Sree Tech Private Limited,श्री टेक Private Limited,shrI Teka Private Limited,907/111Maharaja Aggarsen Market 1St Floor Chaw...,DOOR NO G-907/111MAHARAJA AGGARSEN MARKET 1ST ...,84.000000,93.670886,87.868354
341288,S1-136248523,S2-331656205,Sree Tech Private Limited,श्री टेक प्राइवेट लिमिटेड,shrI Teka prAiveTa limiTeDa,907/111Maharaja Aggarsen Market 1St Floor Chaw...,G-907/111MAHARAJA AGGARSEN MARKET 1ST FLOOR CH...,73.076923,94.871795,81.794872
341289,S1-136248523,S2-548876044,Sree Tech Private Limited,श्री टेक प्राइवेट लिमिटेड,shrI Teka prAiveTa limiTeDa,907/111Maharaja Aggarsen Market 1St Floor Chaw...,G-907/111MAHARAJA AGGARSEN MARKET 1ST FLOOR CH...,73.076923,94.871795,81.794872
341287,S1-136248523,S2-116369112,Sree Tech Private Limited,श्री टेक प्राइवेट लिमिटेड,shrI Teka prAiveTa limiTeDa,907/111Maharaja Aggarsen Market 1St Floor Chaw...,NO G-907/111MAHARAJA AGGARSEN MARKET 1ST FLOOR...,73.076923,93.081761,81.078858
662549,S1-142579261,S2-996810134,Vijay Technologies Private Limited,విజయ్ టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,vijay TèknAlajIs praiveT limiTèD,"12-463/A G1-Doctors Paradise Tadepalle, Guntur...","12-463/A G1-DOCTORS PARADISE TADEPALLE, GUNTUR...",72.727273,100.000000,83.636364
662550,S1-142579261,S2-613770789,Vijay Technologies Private Limited,విజయ్ టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,vijay TèknAlajIs praiveT limiTèD,"12-463/A G1-Doctors Paradise Tadepalle, Guntur...","12-463/A G1-DOCTORS PARADISE TADEPALLE, GUNTUR...",72.727273,100.000000,83.636364


In [160]:
# ============================================================
# STEP 26: CANDIDATE SCORE-GAP ANALYSIS — FINAL FIX
# ============================================================

import pandas as pd
import numpy as np

THRESHOLD = 80

# ------------------------------------------------------------
# 1. Keep qualified candidates
# ------------------------------------------------------------

qualified = cross_script[
    cross_script["combined_score_translit"] >= THRESHOLD
].copy()

# ------------------------------------------------------------
# 2. Sort candidates from highest to lowest score
# ------------------------------------------------------------

qualified = qualified.sort_values(
    ["source1_entity_id", "combined_score_translit"],
    ascending=[True, False]
).copy()

# ------------------------------------------------------------
# 3. Assign candidate rank
# ------------------------------------------------------------

qualified["candidate_rank"] = (
    qualified
    .groupby("source1_entity_id")
    .cumcount() + 1
)

# ------------------------------------------------------------
# 4. Create one row per S1 entity
#    Columns = candidate ranks
# ------------------------------------------------------------

score_table = qualified.pivot_table(
    index="source1_entity_id",
    columns="candidate_rank",
    values="combined_score_translit",
    aggfunc="first"
)

# Rename candidate columns
score_table = score_table.rename(
    columns={
        1: "best_score",
        2: "second_score",
        3: "third_score",
        4: "fourth_score",
        5: "fifth_score"
    }
)

# ------------------------------------------------------------
# 5. Candidate count
# ------------------------------------------------------------

candidate_counts = (
    qualified
    .groupby("source1_entity_id")
    .size()
    .rename("candidate_count")
)

# ------------------------------------------------------------
# 6. Combine
# ------------------------------------------------------------

ambiguity = score_table.join(candidate_counts)

# ------------------------------------------------------------
# 7. Score gap
# ------------------------------------------------------------

ambiguity["score_gap"] = (
    ambiguity["best_score"] -
    ambiguity["second_score"]
)

print("=" * 100)
print("CROSS-SCRIPT CANDIDATE SCORE-GAP ANALYSIS")
print("=" * 100)

print("\nShape:", ambiguity.shape)

print("\nCandidate-count distribution:")
print(
    ambiguity["candidate_count"]
    .value_counts()
    .sort_index()
)

print("\nScore-gap statistics — multiple candidates only:")

multiple = ambiguity[
    ambiguity["candidate_count"] > 1
].copy()

print(
    multiple["score_gap"]
    .describe()
)

# ------------------------------------------------------------
# 8. Score-gap bands
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SCORE-GAP BANDS")
print("=" * 100)

gap_bins = [-0.0001, 1, 2, 5, 10, 20, 1000]

gap_labels = [
    "0-1",
    "1-2",
    "2-5",
    "5-10",
    "10-20",
    "20+"
]

gap_distribution = pd.cut(
    multiple["score_gap"],
    bins=gap_bins,
    labels=gap_labels,
    include_lowest=True
).value_counts().sort_index()

print(gap_distribution)

# ------------------------------------------------------------
# 9. Exact tied candidates
# ------------------------------------------------------------

exact_ties = multiple[
    multiple["score_gap"] == 0
]

print("\nExact ties:")
print(len(exact_ties))

# ------------------------------------------------------------
# 10. Most ambiguous S1 entities
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MOST AMBIGUOUS S1 ENTITIES")
print("=" * 100)

most_ambiguous_ids = (
    multiple
    .sort_values(
        ["score_gap", "candidate_count"],
        ascending=[True, False]
    )
    .head(20)
    .index
)

display(
    qualified[
        qualified["source1_entity_id"]
        .isin(most_ambiguous_ids)
    ][
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit",
            "candidate_rank"
        ]
    ]
    .sort_values(
        ["source1_entity_id", "candidate_rank"]
    )
)

# ------------------------------------------------------------
# 11. Consistency checks
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CONSISTENCY CHECK")
print("=" * 100)

print(
    "Qualified candidate rows:",
    len(qualified)
)

print(
    "Unique S1 entities:",
    qualified["source1_entity_id"].nunique()
)

print(
    "Ambiguous S1 entities:",
    len(multiple)
)

print(
    "Maximum candidates:",
    int(ambiguity["candidate_count"].max())
)

print(
    "Score gaps available:",
    ambiguity["score_gap"].notna().sum()
)

CROSS-SCRIPT CANDIDATE SCORE-GAP ANALYSIS

Shape: (50811, 7)

Candidate-count distribution:
candidate_count
1    42206
2     7691
3      850
4       61
5        3
Name: count, dtype: int64

Score-gap statistics — multiple candidates only:
count    8605.000000
mean        2.141611
std         2.935199
min         0.000000
25%         0.000000
50%         0.817050
75%         3.448276
max        19.104478
Name: score_gap, dtype: float64

SCORE-GAP BANDS
score_gap
0-1      4511
1-2       851
2-5      1975
5-10     1029
10-20     239
20+         0
Name: count, dtype: int64

Exact ties:
3460

MOST AMBIGUOUS S1 ENTITIES


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit,candidate_rank
662549,S1-142579261,S2-996810134,Vijay Technologies Private Limited,విజయ్ టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,vijay TèknAlajIs praiveT limiTèD,"12-463/A G1-Doctors Paradise Tadepalle, Guntur...","12-463/A G1-DOCTORS PARADISE TADEPALLE, GUNTUR...",72.727273,100.000000,83.636364,1
662550,S1-142579261,S2-613770789,Vijay Technologies Private Limited,విజయ్ టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,vijay TèknAlajIs praiveT limiTèD,"12-463/A G1-Doctors Paradise Tadepalle, Guntur...","12-463/A G1-DOCTORS PARADISE TADEPALLE, GUNTUR...",72.727273,100.000000,83.636364,2
662551,S1-142579261,S2-189669243,Vijay Technologies Private Limited,విజయ్ టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,vijay TèknAlajIs praiveT limiTèD,"12-463/A G1-Doctors Paradise Tadepalle, Guntur...","#12-463/A G1-DOCTORS PARADISE TADEPALLE, GUNTU...",72.727273,100.000000,83.636364,3
662552,S1-142579261,S2-736174563,Vijay Technologies Private Limited,విజయ్ టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,vijay TèknAlajIs praiveT limiTèD,"12-463/A G1-Doctors Paradise Tadepalle, Guntur...","12-463/A G1-DOCTORS PARADISE TADEPALLE, GUNTUR...",72.727273,100.000000,83.636364,4
500692,S1-210692425,S2-136971823,Bharat Investments,भारत इन्वेस्टमेंट्स,bhArata invesTameMTsa,"A-5Plot-5A Jindal Mansion, Dr Gopalrao Deshmuk...","A-5PLOT-5A JINDAL MANSION, DR GOPALRAO DESHMUK...",87.179487,100.000000,92.307692,1
...,...,...,...,...,...,...,...,...,...,...,...
377527,S1-82443192,S2-577124023,East Guru Food Private Limited,ಈಸ್ಟ್ ಗುರು ಫುಡ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,IsT guru phuD praiveT limiTèD,"No.59/3, 21St A Main, 16Th Cross Sector 1, Hsr...","NO 59/3, 21ST A MAIN, 16TH CROSS SECTOR 1, HSR...",74.576271,100.000000,84.745763,1
377530,S1-82443192,S2-914863960,East Guru Food Private Limited,ಈಸ್ಟ್ ಗುರು ಫುಡ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,IsT guru phuD praiveT limiTèD,"No.59/3, 21St A Main, 16Th Cross Sector 1, Hsr...","NO.59/3, 21ST A MAIN, 16TH CROSS SECTOR 1, HSR...",74.576271,100.000000,84.745763,2
377529,S1-82443192,S2-567703573,East Guru Food Private Limited,ಈಸ್ಟ್ ಗುರು ಫುಡ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,IsT guru phuD praiveT limiTèD,"No.59/3, 21St A Main, 16Th Cross Sector 1, Hsr...","H.NO 59/3, 21ST A MAIN, 16TH CROSS SECTOR 1, H...",74.576271,98.591549,84.182382,3
377528,S1-82443192,S2-13233676,East Guru Food Private Limited,ಈಸ್ಟ್ ಗುರು ಫುಡ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,IsT guru phuD praiveT limiTèD,"No.59/3, 21St A Main, 16Th Cross Sector 1, Hsr...","NO 59/3, 21ST A MAIN, 16TH CROSS SECTOR 1, HSR...",74.576271,88.405797,80.108082,4



CONSISTENCY CHECK
Qualified candidate rows: 60397
Unique S1 entities: 50811
Ambiguous S1 entities: 8605
Maximum candidates: 5
Score gaps available: 8605


In [163]:
# Show dataframes currently available that contain candidate information
for name, obj in globals().items():
    if hasattr(obj, "columns"):
        cols = list(obj.columns)
        if "candidate_rank" in cols or "candidate_count" in cols:
            print(name, "->", obj.shape)
            print(cols)
            print()

v1_candidate_pairs -> (129478, 4)
['s1_count', 's2_count', 's3_count', 'candidate_count']

qualified -> (60397, 27)
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country', 'name_s1_norm', 'name_s2_norm', 'address_s1_norm', 'address_s2_norm', 'name_score', 'address_score', 'combined_score', 's1_script', 's2_script', 'matched_name_latin', 's1_name_norm', 's2_name_latin_norm', 'name_score_translit', 'combined_score_translit', 'matched_name_latin_norm', 'candidate_rank']

multiple_candidates -> (8605, 2)
['source1_entity_id', 'candidate_count']

ambiguity -> (50811, 7)
['best_score', 'second_score', 'third_score', 'fourth_score', 'fifth_score', 'candidate_count', 'score_gap']

ambiguous -> (8605, 4)
['best_score', 'second_score', 'candidate_count', 'score_gap']

multiple -> (8605, 7)
['best_score', 'second_score', 'third_score', 'fourth_score', 'fifth_score', 'c

In [164]:
# ============================================================
# STEP 28: AMBIGUOUS CANDIDATE FEATURE ANALYSIS
# ============================================================

print("=" * 100)
print("STEP 28 — AMBIGUOUS CANDIDATE FEATURE ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# 1. Get candidate counts from qualified
# ------------------------------------------------------------

candidate_counts = (
    qualified
    .groupby("source1_entity_id")
    .size()
    .rename("candidate_count")
)

# ------------------------------------------------------------
# 2. Keep only S1 entities having multiple candidates
# ------------------------------------------------------------

ambiguous_candidates = qualified[
    qualified["source1_entity_id"].isin(
        candidate_counts[candidate_counts > 1].index
    )
].copy()

# ------------------------------------------------------------
# 3. Calculate score gap
# ------------------------------------------------------------

score_ranked = (
    ambiguous_candidates
    .sort_values(
        ["source1_entity_id", "combined_score_translit"],
        ascending=[True, False]
    )
    .copy()
)

score_ranked["candidate_rank"] = (
    score_ranked
    .groupby("source1_entity_id")
    .cumcount() + 1
)

best_scores = (
    score_ranked
    .groupby("source1_entity_id")["combined_score_translit"]
    .agg(
        best_score="first",
        second_score=lambda x: x.iloc[1] if len(x) > 1 else None
    )
)

best_scores["score_gap"] = (
    best_scores["best_score"] -
    best_scores["second_score"]
)

# ------------------------------------------------------------
# 4. Attach gap information
# ------------------------------------------------------------

score_ranked = score_ranked.merge(
    best_scores[["best_score", "second_score", "score_gap"]],
    left_on="source1_entity_id",
    right_index=True,
    how="left"
)

print("\nAmbiguous S1 entities:",
      score_ranked["source1_entity_id"].nunique())

# ------------------------------------------------------------
# 5. Score-gap statistics
# ------------------------------------------------------------

print("\nScore-gap statistics — multiple candidates only:")

print(
    best_scores["score_gap"].describe()
)

# ------------------------------------------------------------
# 6. Score-gap bands
# ------------------------------------------------------------

def gap_band(x):
    if x < 1:
        return "0-1"
    elif x < 2:
        return "1-2"
    elif x < 5:
        return "2-5"
    elif x < 10:
        return "5-10"
    elif x < 20:
        return "10-20"
    else:
        return "20+"

gap_bands = (
    best_scores["score_gap"]
    .apply(gap_band)
    .value_counts()
    .reindex(
        ["0-1", "1-2", "2-5", "5-10", "10-20", "20+"],
        fill_value=0
    )
)

print("\n" + "=" * 100)
print("SCORE-GAP BANDS")
print("=" * 100)

print(gap_bands)

# ------------------------------------------------------------
# 7. Exact ties
# ------------------------------------------------------------

exact_ties_count = (
    best_scores["score_gap"] == 0
).sum()

print("\nExact ties:")
print(exact_ties_count)

# ------------------------------------------------------------
# 8. Hardest ambiguous cases
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HARDEST AMBIGUOUS CASES — GAP < 1")
print("=" * 100)

hard_ambiguous = score_ranked[
    score_ranked["score_gap"] < 1
].copy()

print(
    "S1 entities with gap < 1:",
    hard_ambiguous["source1_entity_id"].nunique()
)

display(
    hard_ambiguous[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit",
            "candidate_rank",
            "score_gap"
        ]
    ]
    .sort_values(
        ["source1_entity_id", "candidate_rank"]
    )
    .head(150)
)

STEP 28 — AMBIGUOUS CANDIDATE FEATURE ANALYSIS

Ambiguous S1 entities: 8605

Score-gap statistics — multiple candidates only:
count    8605.000000
mean        2.141611
std         2.935199
min         0.000000
25%         0.000000
50%         0.817050
75%         3.448276
max        19.104478
Name: score_gap, dtype: float64

SCORE-GAP BANDS
score_gap
0-1      4507
1-2       850
2-5      1972
5-10     1034
10-20     242
20+         0
Name: count, dtype: int64

Exact ties:
3460

HARDEST AMBIGUOUS CASES — GAP < 1
S1 entities with gap < 1: 4507


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit,candidate_rank,score_gap
152834,S1-100449828,S2-100119491,Hotel Care Private Limited,होटल केयर प्राइवेट लिमिटेड,hoTala keyara prAiveTa limiTeDa,"Office No-1104, G Square Business Park, Plot N...","OFFICE NO-1104, G SQUARE BUSINESS PARK, PLOT N...",73.684211,90.625000,80.460526,1,0.000000
152835,S1-100449828,S2-226476559,Hotel Care Private Limited,होटल केयर प्राइवेट लिमिटेड,hoTala keyara prAiveTa limiTeDa,"Office No-1104, G Square Business Park, Plot N...","OFFICE NO-1104, G SQUARE BUSINESS PARK, PLOT N...",73.684211,90.625000,80.460526,2,0.000000
600591,S1-100469064,S2-512775597,First Investment Private Limited,फर्स्ट इन्वेस्टमेंट प्राइवेट लिमिटेड,pharsTa invesTameMTa prAiveTa limiTeDa,"711 Narayan Pethkaxmi Road, Pune, Maharashtra","711 NARAYAN PETHKAXMI ROAD, PUNE, Maharashtra",77.142857,100.000000,86.285714,1,0.470588
600593,S1-100469064,S2-48744415,First Investment Private Limited,फर्स्ट इन्वेस्टमेंट प्राइवेट लिमिटेड,pharsTa invesTameMTa prAiveTa limiTeDa,"711 Narayan Pethkaxmi Road, Pune, Maharashtra","711 NARAYAN PETHKAXMI ROD, PUNE, Maharashtra",77.142857,98.823529,85.815126,2,0.470588
283404,S1-101107998,S2-643645483,Sky Shakti Industries Private Limited,स्काई शक्ति इंडस्ट्रीज प्राइवेट लिमिटेड,skAI shakti iMDasTrIja prAiveTa limiTeDa,"C/O.Babalya Vithal Jadhav, A/Pkolape Tal. Boud...","C/O.BABALYA VITHAL JADHAV, A/PKOLAPE TAL. BOUD...",77.922078,86.904762,81.515152,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
439867,S1-115699208,S2-898714572,My Trading Private Limited,माय ट्रेडिंग प्राइवेट लिमिटेड,mAya TreDiMga prAiveTa limiTeDa,"Plot No. Rz -8-A, F/F, Kh. No. 59/9, 13, 19 & ...","Delhi, PLOT NO. RZ -8-A, F/F, KH. NO. 59/9, 13...",77.192982,93.750000,83.815789,2,0.000000
145223,S1-115781627,S2-630037767,Gold Foundation Private Limited,गोल्ड फाउंडेशन प्राइवेट लिमिटेड,golDa phAuMDeshana prAiveTa limiTeDa,"250, Chuna Bhatti, Dudhia, Indore, Madhya Pradesh","250, CHUNA BHATTI, DUDHIA, INDORE, Madhya Pradesh",68.656716,100.000000,81.194030,1,0.000000
145224,S1-115781627,S2-204575964,Gold Foundation Private Limited,गोल्ड फाउंडेशन प्राइवेट लिमिटेड,golDa phAuMDeshana prAiveTa limiTeDa,"250, Chuna Bhatti, Dudhia, Indore, Madhya Pradesh","250, CHUNA BHATTI, DUDHIA, INDORE, Madhya Pradesh",68.656716,100.000000,81.194030,2,0.000000
369016,S1-115788461,S2-70497646,East Media Private Limited,ईस्ट मीडिया प्राइवेट लिमिटेड,IsTa mIDiyA prAiveTa limiTeDa,"Akbar Villa Qayyum Nagar, Moradabad, Uttar Pra...","AKBAR VILLA QAYYUM NAGAR, MORADABAD, Uttar Pra...",76.363636,100.000000,85.818182,1,0.833333


In [167]:
# ============================================================
# STEP 29: CROSS-SCRIPT RANK QUALITY ANALYSIS
# ============================================================

print("=" * 100)
print("STEP 29 — CROSS-SCRIPT RANK QUALITY ANALYSIS")
print("=" * 100)

df = score_ranked.copy()

# ------------------------------------------------------------
# 1. True / false candidate counts per S1
# ------------------------------------------------------------

true_counts = (
    df.groupby("source1_entity_id")["captured"]
      .sum()
      .rename("true_candidate_count")
)

df = df.merge(
    true_counts,
    left_on="source1_entity_id",
    right_index=True,
    how="left"
)

print("\nTrue-candidate distribution among ambiguous S1 entities:")
print(
    true_counts.value_counts().sort_index()
)

# ------------------------------------------------------------
# 2. Is the top-ranked candidate true?
# ------------------------------------------------------------

top1 = df[df["candidate_rank"] == 1].copy()

top1_true = top1["captured"].sum()
top1_total = len(top1)

print("\n" + "=" * 100)
print("TOP-1 PERFORMANCE")
print("=" * 100)

print("Ambiguous S1 entities:", top1_total)
print("Top-1 true:", int(top1_true))
print(
    "Top-1 accuracy: {:.2f}%".format(
        100 * top1_true / top1_total
    )
)

# ------------------------------------------------------------
# 3. True candidate by rank
# ------------------------------------------------------------

rank_analysis = (
    df.groupby("candidate_rank")
      .agg(
          candidates=("captured", "size"),
          true_candidates=("captured", "sum")
      )
)

rank_analysis["true_rate_%"] = (
    100 *
    rank_analysis["true_candidates"] /
    rank_analysis["candidates"]
)

print("\n" + "=" * 100)
print("TRUE RATE BY CANDIDATE RANK")
print("=" * 100)

print(rank_analysis)

# ------------------------------------------------------------
# 4. Top candidate vs whether S1 has multiple true matches
# ------------------------------------------------------------

# Make sure top1 has candidate_count
# candidate_count is the number of qualified candidates for each S1 entity
candidate_counts = (
    qualified
    .groupby("source1_entity_id")
    .size()
    .rename("candidate_count")
    .reset_index()
)

# Count TRUE matches for each S1 entity
true_match_counts = (
    qualified
    .groupby("source1_entity_id")["captured"]
    .sum()
    .rename("true_match_count")
    .reset_index()
)

# Add candidate_count and true_match_count to top1
top1_analysis = (
    top1
    .drop(columns=["candidate_count", "true_match_count"], errors="ignore")
    .merge(candidate_counts, on="source1_entity_id", how="left")
    .merge(true_match_counts, on="source1_entity_id", how="left")
)

# Fill missing values safely
top1_analysis["candidate_count"] = (
    top1_analysis["candidate_count"]
    .fillna(0)
    .astype(int)
)

top1_analysis["true_match_count"] = (
    top1_analysis["true_match_count"]
    .fillna(0)
    .astype(int)
)

# Flag whether S1 has multiple true matches
top1_analysis["multiple_true_matches"] = (
    top1_analysis["true_match_count"] > 1
)

print("\n" + "=" * 100)
print("TOP-1 CANDIDATE VS TRUE MATCH COUNT")
print("=" * 100)

print(
    "S1 entities with multiple candidates:",
    (top1_analysis["candidate_count"] > 1).sum()
)

print(
    "S1 entities with multiple true matches:",
    top1_analysis["multiple_true_matches"].sum()
)

# ------------------------------------------------------------
# Display top-1 analysis
# ------------------------------------------------------------

display(
    top1_analysis[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit",
            "candidate_count",
            "true_match_count",
            "multiple_true_matches"
        ]
    ]
    .sort_values(
        ["candidate_count", "combined_score_translit"],
        ascending=[False, False]
    )
    .head(150)
)
# ------------------------------------------------------------
# 5. Exact ties
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("EXACT-TIE ANALYSIS")
print("=" * 100)

exact_tie_s1 = best_scores[
    best_scores["score_gap"] == 0
].index

ties = df[
    df["source1_entity_id"].isin(exact_tie_s1)
].copy()

tie_summary = (
    ties.groupby("source1_entity_id")
        .agg(
            candidate_count=("matched_entity_ids", "size"),
            true_count=("captured", "sum")
        )
)

tie_summary["all_candidates_true"] = (
    tie_summary["true_count"] ==
    tie_summary["candidate_count"]
)

tie_summary["no_candidates_true"] = (
    tie_summary["true_count"] == 0
)

tie_summary["partial_true"] = (
    (tie_summary["true_count"] > 0) &
    (tie_summary["true_count"] <
     tie_summary["candidate_count"])
)

print("\nExact-tie S1 entities:", len(tie_summary))

print(
    "\nAll tied candidates true:",
    int(tie_summary["all_candidates_true"].sum())
)

print(
    "No tied candidates true:",
    int(tie_summary["no_candidates_true"].sum())
)

print(
    "Some tied candidates true:",
    int(tie_summary["partial_true"].sum())
)

# ------------------------------------------------------------
# 6. Show partial ties
# ------------------------------------------------------------

partial_tie_ids = tie_summary[
    tie_summary["partial_true"]
].index

partial_ties = ties[
    ties["source1_entity_id"].isin(partial_tie_ids)
].copy()

print("\n" + "=" * 100)
print("PARTIAL EXACT TIES — MOST IMPORTANT CASES")
print("=" * 100)

display(
    partial_ties[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit",
            "candidate_rank",
            "captured"
        ]
    ]
    .sort_values(
        ["source1_entity_id", "candidate_rank"]
    )
    .head(150)
)

STEP 29 — CROSS-SCRIPT RANK QUALITY ANALYSIS

True-candidate distribution among ambiguous S1 entities:
true_candidate_count
0    8605
Name: count, dtype: int64

TOP-1 PERFORMANCE
Ambiguous S1 entities: 8605
Top-1 true: 0
Top-1 accuracy: 0.00%

TRUE RATE BY CANDIDATE RANK
                candidates  true_candidates  true_rate_%
candidate_rank                                          
1                     8605                0          0.0
2                     8605                0          0.0
3                      914                0          0.0
4                       64                0          0.0
5                        3                0          0.0

TOP-1 CANDIDATE VS TRUE MATCH COUNT
S1 entities with multiple candidates: 8605
S1 entities with multiple true matches: 0


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit,candidate_count,true_match_count,multiple_true_matches
4228,S1-540969303,S2-864911434,Shiv Super Media Private Limited,शिव सुपर मीडिया प्राइवेट लिमिटेड,shiva supara mIDiyA prAiveTa limiTeDa,C/O Nabila Arshad R/O A-44 Third Floor Shaheen...,C/O NABILA ARSHAD R/O A-44 THIRD FLOOR SHAHEEN...,81.159420,100.000000,88.695652,5,0,False
1367,S1-246004420,S2-673770047,Om Hospitality Private Limited,ओम Hospitality Private Limited,oma Hospitality Private Limited,"Vivek Industrial Estate, Gala 17, Walbhat Rd, ...","महाराष्ट्र, null, MUMBAI, VIVEK INDUSTRIAL EST...",98.360656,67.469880,86.004345,5,0,False
6918,S1-82443192,S2-577124023,East Guru Food Private Limited,ಈಸ್ಟ್ ಗುರು ಫುಡ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,IsT guru phuD praiveT limiTèD,"No.59/3, 21St A Main, 16Th Cross Sector 1, Hsr...","NO 59/3, 21ST A MAIN, 16TH CROSS SECTOR 1, HSR...",74.576271,100.000000,84.745763,5,0,False
3180,S1-433054591,S2-220418946,Tirupati Hospitality Limited,తిరుపతి Hospitality లిమిటెడ్,tirupati Hospitality limiTèD,"6-3-898 Somajiguda, Hyderabad, Telangana","6-3-898 SOMAJIGUDA, HYDERABAD, Telangana",96.428571,100.000000,97.857143,4,0,False
1943,S1-303887578,S2-327902918,Al Projects Private Limited,अल प्रोजेक्ट्स Private Limited,ala projekTsa Private Limited,"Flat No.. 301, Gh.No.-7, Sec-45, Sector -45, G...","FLAT NO.. 301, GH.NO.-7, SEC-45, SECTOR -45, G...",92.857143,100.000000,95.714286,4,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6066,S1-734317100,S2-508044149,Jay Apex Investments Limited,जय एपेक्स Investments लिमिटेड,jaya epeksa Investments limiTeDa,"B 34 Ist Floorlaxmi Nagar Vikas Marg, Delhi, E...","B 34. IST FLOORLAXMI NAGAR VIKAS MARG, DELHI, ...",86.666667,100.000000,92.000000,3,0,False
7887,S1-926096865,S2-229741669,Shyam Global Private Limited,श्याम ग्लोबल प्राइवेट लिमिटेड,shyAma globala prAiveTa limiTeDa,"Khs-260 Green Park Colony, Jalpura, Dadri, Gau...","KHS-260 GREEN PARK COLONY, JALPURA, DADRI, GAU...",86.666667,100.000000,92.000000,3,0,False
2255,S1-335685499,S2-429530928,Balaji Marketing Private Limited,बालाजी मार्केटिंग प्राइवेट लिमिटेड,bAlAjI mArkeTiMga prAiveTa limiTeDa,"81 Electric Dp Ke Pass Soni Colony, Guna, Madh...","81 ELECTRIC DP KE PASS SONI COLONY, GUNA, Madh...",86.567164,100.000000,91.940299,3,0,False
8041,S1-941656647,S2-101885606,Vision Energy Private Limited,विजन Energy Private Limited,vijana Energy Private Limited,H No. 165 K Sighi-2 Singhi Thana Sahatwar Teh ...,DOOR NO 16 K SIGHI-2 SINGHI THANA SAHATWR TEH ...,89.655172,95.238095,91.888342,3,0,False



EXACT-TIE ANALYSIS

Exact-tie S1 entities: 3460

All tied candidates true: 0
No tied candidates true: 3460
Some tied candidates true: 0

PARTIAL EXACT TIES — MOST IMPORTANT CASES


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit,candidate_rank,captured


In [169]:
[x for x in globals().keys()
 if any(k in x.lower() for k in ["true", "match", "ground", "label", "pair"])]

['gt_pairs',
 'total_pairs',
 'captured_pairs',
 'missed_pairs',
 'v1_candidate_pairs',
 'empty_postal_match',
 'block_mismatch',
 'true_s2',
 'true_s2_scored',
 'SequenceMatcher',
 'true_captured',
 'true_recall',
 'true_cross_script',
 'total_true',
 'normalize_for_match',
 'labels',
 'true_bands',
 'true_counts',
 'true_mask',
 'gap_labels',
 'top1_true',
 'true_match_counts',
 'all_tied_true',
 'no_tied_true',
 'some_tied_true']

In [170]:
# ============================================================
# INSPECT GROUND-TRUTH STRUCTURE
# ============================================================

print("gt_pairs shape:", gt_pairs.shape)
print("gt_pairs columns:")
print(gt_pairs.columns.tolist())

print("\nFirst 10 ground-truth rows:")
display(gt_pairs.head(10))

print("\nqualified columns:")
print(qualified.columns.tolist())

gt_pairs shape: (7638365, 3)
gt_pairs columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key']

First 10 ground-truth rows:


,source1_entity_id,matched_entity_ids,s1_block_key
0,S1-965667,S2-681193310,US_maur
1,S1-965667,S2-743505751,US_maur
2,S1-965667,S3-775321672,US_maur
3,S1-965667,S3-11291185,US_maur
4,S1-965667,S3-860443364,US_maur
5,S1-55344266,S2-249013014,India_raji
6,S1-55344266,S2-197070651,India_raji
7,S1-55344266,S3-478195123,India_raji
8,S1-55344266,S3-384364074,India_raji
9,S1-343815751,S2-790675320,US_dahl



qualified columns:
['source1_entity_id', 'matched_entity_ids', 's1_block_key', 'matched_block_key', 'captured', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country', 'name_s1_norm', 'name_s2_norm', 'address_s1_norm', 'address_s2_norm', 'name_score', 'address_score', 'combined_score', 's1_script', 's2_script', 'matched_name_latin', 's1_name_norm', 's2_name_latin_norm', 'name_score_translit', 'combined_score_translit', 'matched_name_latin_norm', 'candidate_rank']


In [171]:
# ============================================================
# CHECK WHETHER GT PAIRS CAN BE MATCHED TO QUALIFIED
# ============================================================

print("Ground truth rows:", len(gt_pairs))
print("Qualified rows:", len(qualified))

# Show likely ID columns
print("\nGT dtypes:")
print(gt_pairs.dtypes)

print("\nQualified dtypes:")
print(
    qualified[
        ["source1_entity_id", "matched_entity_ids"]
    ].dtypes
)

Ground truth rows: 7638365
Qualified rows: 60397

GT dtypes:
source1_entity_id     str
matched_entity_ids    str
s1_block_key          str
dtype: object

Qualified dtypes:
source1_entity_id     str
matched_entity_ids    str
dtype: object


In [172]:
# ============================================================
# STEP 29 — CROSS-SCRIPT RANK QUALITY ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("STEP 29 — CROSS-SCRIPT RANK QUALITY ANALYSIS")
print("=" * 100)


# ------------------------------------------------------------
# 1. BUILD GROUND-TRUTH PAIR SET
# ------------------------------------------------------------

gt_pair_keys = pd.MultiIndex.from_frame(
    gt_pairs[
        ["source1_entity_id", "matched_entity_ids"]
    ]
)

qualified_pair_keys = pd.MultiIndex.from_frame(
    qualified[
        ["source1_entity_id", "matched_entity_ids"]
    ]
)

qualified = qualified.copy()

qualified["is_true"] = (
    qualified_pair_keys.isin(gt_pair_keys)
)

print("\nGround-truth rows:", len(gt_pairs))
print("Qualified candidate rows:", len(qualified))
print("True qualified candidates:", qualified["is_true"].sum())
print(
    "False qualified candidates:",
    (~qualified["is_true"]).sum()
)


# ------------------------------------------------------------
# 2. TRUE-CANDIDATE COUNT PER S1
# ------------------------------------------------------------

true_match_counts = (
    qualified
    .groupby("source1_entity_id")["is_true"]
    .sum()
    .rename("true_candidate_count")
)

candidate_counts = (
    qualified
    .groupby("source1_entity_id")
    .size()
    .rename("candidate_count")
)

entity_summary = pd.concat(
    [candidate_counts, true_match_counts],
    axis=1
).fillna(0)

entity_summary["candidate_count"] = (
    entity_summary["candidate_count"].astype(int)
)

entity_summary["true_candidate_count"] = (
    entity_summary["true_candidate_count"].astype(int)
)

entity_summary["multiple_true_matches"] = (
    entity_summary["true_candidate_count"] > 1
)


# ------------------------------------------------------------
# 3. AMBIGUOUS S1 ENTITIES
# ------------------------------------------------------------

ambiguous_ids = entity_summary.index[
    entity_summary["candidate_count"] > 1
]

ambiguous = entity_summary.loc[
    ambiguous_ids
].copy()

print("\n" + "=" * 100)
print("TRUE-CANDIDATE DISTRIBUTION AMONG AMBIGUOUS S1 ENTITIES")
print("=" * 100)

print(
    ambiguous["true_candidate_count"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. TOP-1 PERFORMANCE
# ------------------------------------------------------------

top1 = (
    qualified[
        qualified["candidate_rank"] == 1
    ]
    .copy()
)

top1_true = top1["is_true"].sum()
top1_total = len(top1)

top1_accuracy = (
    100 * top1_true / top1_total
    if top1_total > 0 else 0
)

print("\n" + "=" * 100)
print("TOP-1 PERFORMANCE")
print("=" * 100)

print("Ambiguous S1 entities:", len(ambiguous))
print("Top-1 true:", int(top1_true))
print(f"Top-1 accuracy: {top1_accuracy:.2f}%")


# ------------------------------------------------------------
# 5. TRUE RATE BY CANDIDATE RANK
# ------------------------------------------------------------

rank_quality = (
    qualified
    .groupby("candidate_rank")
    .agg(
        candidates=("matched_entity_ids", "size"),
        true_candidates=("is_true", "sum")
    )
)

rank_quality["true_rate_%"] = (
    100
    * rank_quality["true_candidates"]
    / rank_quality["candidates"]
)

print("\n" + "=" * 100)
print("TRUE RATE BY CANDIDATE RANK")
print("=" * 100)

display(rank_quality)


# ------------------------------------------------------------
# 6. TOP-1 CANDIDATE VS TRUE MATCH COUNT
# ------------------------------------------------------------

top1_analysis = (
    qualified[
        qualified["candidate_rank"] == 1
    ][
        [
            "source1_entity_id",
            "matched_entity_ids",
            "s1_name",
            "matched_name",
            "matched_name_latin",
            "s1_address",
            "matched_address",
            "name_score_translit",
            "address_score",
            "combined_score_translit",
            "candidate_rank",
            "is_true"
        ]
    ]
    .merge(
        entity_summary[
            [
                "candidate_count",
                "true_candidate_count",
                "multiple_true_matches"
            ]
        ],
        left_on="source1_entity_id",
        right_index=True,
        how="left"
    )
)

print("\n" + "=" * 100)
print("TOP-1 CANDIDATE VS TRUE MATCH COUNT")
print("=" * 100)

print(
    "S1 entities with multiple candidates:",
    len(ambiguous)
)

print(
    "S1 entities with multiple true matches:",
    int(
        ambiguous["multiple_true_matches"].sum()
    )
)

display(
    top1_analysis
    .sort_values(
        ["candidate_count", "combined_score_translit"],
        ascending=[False, False]
    )
    .head(150)
)


# ------------------------------------------------------------
# 7. EXACT-TIE ANALYSIS
# ------------------------------------------------------------

# Recreate score summary directly from qualified candidates

score_pivot = (
    qualified
    .sort_values(
        ["source1_entity_id", "candidate_rank"]
    )
    .groupby("source1_entity_id")[
        "combined_score_translit"
    ]
    .apply(
        lambda x: list(x)
    )
)

tie_records = []

for s1_id, scores in score_pivot.items():

    if len(scores) < 2:
        continue

    best = scores[0]
    second = scores[1]

    if pd.notna(best) and pd.notna(second):

        tie_records.append(
            {
                "source1_entity_id": s1_id,
                "best_score": best,
                "second_score": second,
                "score_gap": best - second
            }
        )

tie_df = pd.DataFrame(tie_records)

exact_tie_ids = tie_df.loc[
    tie_df["score_gap"] == 0,
    "source1_entity_id"
]

exact_tie_candidates = qualified[
    qualified["source1_entity_id"].isin(
        exact_tie_ids
    )
].copy()

# Only candidates tied at the best score
exact_tie_candidates = exact_tie_candidates[
    exact_tie_candidates["combined_score_translit"]
    ==
    exact_tie_candidates.groupby(
        "source1_entity_id"
    )["combined_score_translit"].transform("max")
]

tie_truth = (
    exact_tie_candidates
    .groupby("source1_entity_id")["is_true"]
    .agg(
        all_tied_true="all",
        no_tied_true=lambda x: not x.any(),
        some_tied_true=lambda x: x.any() and not x.all()
    )
)

print("\n" + "=" * 100)
print("EXACT-TIE ANALYSIS")
print("=" * 100)

print("Exact-tie S1 entities:", len(exact_tie_ids))

if len(tie_truth) > 0:

    print(
        "All tied candidates true:",
        int(tie_truth["all_tied_true"].sum())
    )

    print(
        "No tied candidates true:",
        int(tie_truth["no_tied_true"].sum())
    )

    print(
        "Some tied candidates true:",
        int(tie_truth["some_tied_true"].sum())
    )

else:

    print("No exact ties found.")


# ------------------------------------------------------------
# 8. SCORE-GAP ANALYSIS
# ------------------------------------------------------------

score_gap_analysis = (
    qualified
    .sort_values(
        ["source1_entity_id", "candidate_rank"]
    )
    .groupby("source1_entity_id")
    .agg(
        best_score=(
            "combined_score_translit",
            "first"
        ),
        second_score=(
            "combined_score_translit",
            lambda x:
            x.iloc[1]
            if len(x) > 1
            else np.nan
        ),
        candidate_count=(
            "matched_entity_ids",
            "size"
        )
    )
)

score_gap_analysis["score_gap"] = (
    score_gap_analysis["best_score"]
    -
    score_gap_analysis["second_score"]
)

multiple_gap = score_gap_analysis[
    score_gap_analysis["candidate_count"] > 1
]["score_gap"].dropna()

print("\n" + "=" * 100)
print("SCORE-GAP ANALYSIS")
print("=" * 100)

print(
    multiple_gap.describe()
)


# ------------------------------------------------------------
# 9. GAP BANDS
# ------------------------------------------------------------

gap_bands = pd.cut(
    multiple_gap,
    bins=[-np.inf, 1, 2, 5, 10, 20, np.inf],
    labels=[
        "0-1",
        "1-2",
        "2-5",
        "5-10",
        "10-20",
        "20+"
    ],
    right=False
)

print("\n" + "=" * 100)
print("SCORE-GAP BANDS")
print("=" * 100)

print(gap_bands.value_counts().sort_index())

print(
    "\nExact ties:",
    int((multiple_gap == 0).sum())
)


# ------------------------------------------------------------
# 10. FINAL SANITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("STEP 29 SANITY CHECK")
print("=" * 100)

print(
    "Qualified candidates:",
    len(qualified)
)

print(
    "True candidates:",
    int(qualified["is_true"].sum())
)

print(
    "False candidates:",
    int((~qualified["is_true"]).sum())
)

print(
    "True candidate rate:",
    f"{100 * qualified['is_true'].mean():.4f}%"
)

STEP 29 — CROSS-SCRIPT RANK QUALITY ANALYSIS

Ground-truth rows: 7638365
Qualified candidate rows: 60397
True qualified candidates: 60397
False qualified candidates: 0

TRUE-CANDIDATE DISTRIBUTION AMONG AMBIGUOUS S1 ENTITIES
true_candidate_count
2    7691
3     850
4      61
5       3
Name: count, dtype: int64

TOP-1 PERFORMANCE
Ambiguous S1 entities: 8605
Top-1 true: 50811
Top-1 accuracy: 100.00%

TRUE RATE BY CANDIDATE RANK


,candidates,true_candidates,true_rate_%
candidate_rank,,,
1,50811,50811,100.0
2,8605,8605,100.0
3,914,914,100.0
4,64,64,100.0
5,3,3,100.0



TOP-1 CANDIDATE VS TRUE MATCH COUNT
S1 entities with multiple candidates: 8605
S1 entities with multiple true matches: 8605


,source1_entity_id,matched_entity_ids,s1_name,matched_name,matched_name_latin,s1_address,matched_address,name_score_translit,address_score,combined_score_translit,candidate_rank,is_true,candidate_count,true_candidate_count,multiple_true_matches
187756,S1-540969303,S2-864911434,Shiv Super Media Private Limited,शिव सुपर मीडिया प्राइवेट लिमिटेड,shiva supara mIDiyA prAiveTa limiTeDa,C/O Nabila Arshad R/O A-44 Third Floor Shaheen...,C/O NABILA ARSHAD R/O A-44 THIRD FLOOR SHAHEEN...,81.159420,100.000000,88.695652,1,True,5,5,True
458245,S1-246004420,S2-673770047,Om Hospitality Private Limited,ओम Hospitality Private Limited,oma Hospitality Private Limited,"Vivek Industrial Estate, Gala 17, Walbhat Rd, ...","महाराष्ट्र, null, MUMBAI, VIVEK INDUSTRIAL EST...",98.360656,67.469880,86.004345,1,True,5,5,True
377527,S1-82443192,S2-577124023,East Guru Food Private Limited,ಈಸ್ಟ್ ಗುರು ಫುಡ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,IsT guru phuD praiveT limiTèD,"No.59/3, 21St A Main, 16Th Cross Sector 1, Hsr...","NO 59/3, 21ST A MAIN, 16TH CROSS SECTOR 1, HSR...",74.576271,100.000000,84.745763,1,True,5,5,True
18832,S1-433054591,S2-220418946,Tirupati Hospitality Limited,తిరుపతి Hospitality లిమిటెడ్,tirupati Hospitality limiTèD,"6-3-898 Somajiguda, Hyderabad, Telangana","6-3-898 SOMAJIGUDA, HYDERABAD, Telangana",96.428571,100.000000,97.857143,1,True,4,4,True
264914,S1-303887578,S2-327902918,Al Projects Private Limited,अल प्रोजेक्ट्स Private Limited,ala projekTsa Private Limited,"Flat No.. 301, Gh.No.-7, Sec-45, Sector -45, G...","FLAT NO.. 301, GH.NO.-7, SEC-45, SECTOR -45, G...",92.857143,100.000000,95.714286,1,True,4,4,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234975,S1-734317100,S2-508044149,Jay Apex Investments Limited,जय एपेक्स Investments लिमिटेड,jaya epeksa Investments limiTeDa,"B 34 Ist Floorlaxmi Nagar Vikas Marg, Delhi, E...","B 34. IST FLOORLAXMI NAGAR VIKAS MARG, DELHI, ...",86.666667,100.000000,92.000000,1,True,3,3,True
644806,S1-926096865,S2-229741669,Shyam Global Private Limited,श्याम ग्लोबल प्राइवेट लिमिटेड,shyAma globala prAiveTa limiTeDa,"Khs-260 Green Park Colony, Jalpura, Dadri, Gau...","KHS-260 GREEN PARK COLONY, JALPURA, DADRI, GAU...",86.666667,100.000000,92.000000,1,True,3,3,True
561117,S1-335685499,S2-429530928,Balaji Marketing Private Limited,बालाजी मार्केटिंग प्राइवेट लिमिटेड,bAlAjI mArkeTiMga prAiveTa limiTeDa,"81 Electric Dp Ke Pass Soni Colony, Guna, Madh...","81 ELECTRIC DP KE PASS SONI COLONY, GUNA, Madh...",86.567164,100.000000,91.940299,1,True,3,3,True
395881,S1-941656647,S2-101885606,Vision Energy Private Limited,विजन Energy Private Limited,vijana Energy Private Limited,H No. 165 K Sighi-2 Singhi Thana Sahatwar Teh ...,DOOR NO 16 K SIGHI-2 SINGHI THANA SAHATWR TEH ...,89.655172,95.238095,91.888342,1,True,3,3,True



EXACT-TIE ANALYSIS
Exact-tie S1 entities: 3460
All tied candidates true: 3460
No tied candidates true: 0
Some tied candidates true: 0

SCORE-GAP ANALYSIS
count    8605.000000
mean        2.141611
std         2.935199
min         0.000000
25%         0.000000
50%         0.817050
75%         3.448276
max        19.104478
Name: score_gap, dtype: float64

SCORE-GAP BANDS
score_gap
0-1      4507
1-2       850
2-5      1972
5-10     1034
10-20     242
20+         0
Name: count, dtype: int64

Exact ties: 3460

STEP 29 SANITY CHECK
Qualified candidates: 60397
True candidates: 60397
False candidates: 0
True candidate rate: 100.0000%
